# HuberRidgeAIME — Additional Experiments (Fully Corrected)

This notebook is a **separate, equation-consistent experimental addendum** . It is designed to answer every scientific request in the review while preventing the silent failures identified in the first additional-experiment run.

## Run order

1. **Run the `USER CONFIGURATION` cell immediately below first.**
2. For final results, leave `USER_QUICK_TEST = False` and `USER_FORCE_RECOMPUTE = True`.
3. Run all remaining cells from top to bottom.
4. The last cell performs validation gates and creates a ZIP only if all required experiments succeeded.

No shell/environment-variable setup is required. Environment variables remain available only as an optional override for automated runs.

## Output layout

```text
./output/additional_validation_manuscript_figures/
├── data/
├── tables/
├── figures/
├── logs/
├── README_additional_validation.md
├── additional_validation_reporting_notes.md
├── additional_validation_run_configuration.json
├── additional_validation_validation_report.json
├── additional_validation_artifact_manifest.csv
└── HuberRidgeAIME_Additional_Robustness_Faithfulness_outputs.zip
```


## Supplementary figures generated directly

- `Supplementary_Figure_S11_fixed_black_box.png`
- `Supplementary_Figure_S12_known_ground_truth.png`
- `Supplementary_Figure_S13_zero_operator_control.png`
- `Supplementary_Figure_S14_lambda_delta_sensitivity.png`
- `Supplementary_Figure_S15_lime_treeshap_stress.png`
- `Supplementary_Figure_S16_spectral_diagnostics.png`

Supplementary Figures S1--S10 retain their current roles and are regenerated by `HuberRidgeAIME_Supplementary_Classwise_and_Synthetic_Visuals_CORRECTED.ipynb`. Every final figure is generated from a validated raw CSV in this notebook. Coincident method values are shown with categorical separation or documented display-only offsets; no experimental y-value is altered.


In [1]:
# USER CONFIGURATION — RUN THIS CELL FIRST
# Final-run defaults are already selected. Edit only when intentionally running a smoke test.

USER_OUTPUT_DIR = "./output/additional_validation_manuscript_figures"
USER_QUICK_TEST = False
USER_FORCE_RECOMPUTE = True
USER_AUTO_INSTALL = True
USER_RUN_END_TO_END = True
USER_RUN_LIME_SHAP = True

print("User configuration loaded")
print("  output:", USER_OUTPUT_DIR)
print("  quick test:", USER_QUICK_TEST)
print("  force recompute:", USER_FORCE_RECOMPUTE)
print("  run end-to-end protocol:", USER_RUN_END_TO_END)
print("  run LIME/TreeSHAP:", USER_RUN_LIME_SHAP)


User configuration loaded
  output: ./output/additional_validation_manuscript_figures
  quick test: False
  force recompute: True
  run end-to-end protocol: True
  run LIME/TreeSHAP: True


In [2]:
# %% [configuration and reproducibility]
import os, sys, io, json, math, time, zipfile, platform, warnings, urllib.request, hashlib, textwrap, shutil
import subprocess, importlib.util
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import softmax

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, average_precision_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PIPELINE_VERSION = "additional_validation_inverse_map_2026-08-31_v3_s11_s16"
ADDITIONAL_SEED = 42
np.random.seed(ADDITIONAL_SEED)

def _bool_setting(user_name, env_name, default):
    if user_name in globals():
        return bool(globals()[user_name])
    return bool(int(os.environ.get(env_name, str(int(default)))))

QUICK_TEST = _bool_setting("USER_QUICK_TEST", "HRA_ADDITIONAL_QUICK_TEST", False)
FORCE_RECOMPUTE = _bool_setting("USER_FORCE_RECOMPUTE", "HRA_ADDITIONAL_FORCE_RECOMPUTE", True)
AUTO_INSTALL = _bool_setting("USER_AUTO_INSTALL", "HRA_ADDITIONAL_AUTO_INSTALL", True)
RUN_BASELINES = _bool_setting("USER_RUN_LIME_SHAP", "HRA_ADDITIONAL_RUN_LIME_SHAP", True)
RUN_END_TO_END = _bool_setting("USER_RUN_END_TO_END", "HRA_ADDITIONAL_RUN_END_TO_END", True)

PROJECT_ROOT = Path.cwd()
_default_output = str(PROJECT_ROOT / "output" / "additional_validation")
_output_setting = globals().get(
    "USER_OUTPUT_DIR", os.environ.get("HRA_ADDITIONAL_OUTPUT_DIR", _default_output)
)
# Never mix a smoke test with final outputs unless the user explicitly supplied a quick path.
if QUICK_TEST and str(_output_setting).rstrip("/").endswith("additional_validation"):
    _output_setting = str(_output_setting).rstrip("/") + "_quick"
ADDITIONAL_OUTDIR = Path(_output_setting).resolve()
ADDITIONAL_DATADIR = ADDITIONAL_OUTDIR / "data"
ADDITIONAL_TABLEDIR = ADDITIONAL_OUTDIR / "tables"
ADDITIONAL_FIGDIR = ADDITIONAL_OUTDIR / "figures"
ADDITIONAL_LOGDIR = ADDITIONAL_OUTDIR / "logs"
for p in [ADDITIONAL_OUTDIR, ADDITIONAL_DATADIR, ADDITIONAL_TABLEDIR, ADDITIONAL_FIGDIR, ADDITIONAL_LOGDIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASETS = ["breast_cancer", "credit_approval", "har"]
LEARNERS = ["lgbm", "svm", "mlp"]
MAX_N_PER_DATASET = {"breast_cancer": None, "credit_approval": None, "har": 3000}
MAX_CLONES = 100

DEFAULT_RIDGE_LAMBDA = 1e-2
DEFAULT_HUBER_DELTA = 1.0
RESIDUAL_SCALE_MODE = "rms"  # ||x_i - A y_i||_2 / sqrt(d); explicitly reported
TOP_K = 10

# Experiment 1A: fixed-black-box inverse-map contamination.
ADDITIONAL_REPEATS = 5
# Experiment 1B: original end-to-end stressed-data protocol.
E2E_REPEATS = 3
ADDITIONAL_OUTLIER_LEVELS = [0.05, 0.10]
ADDITIONAL_CLONE_FRACS = [0.00, 0.25, 0.50]
ADDITIONAL_BOOTSTRAPS = 10
ADDITIONAL_CI_RESAMPLES = 10000
IRRELEVANT_DECOY_TRIALS = 5
IRRELEVANT_DECOY_COUNT = 3

# Experiment 2: known-ground-truth recovery.
SYN_REPEATS = 20
SYN_N = 1400
SYN_D = 100
SYN_K = 10
SYN_RHOS = [0.00, 0.70, 0.95]
SYN_OUTLIER_LEVELS = [0.00, 0.05, 0.10]

# Experiment 3: lambda/delta sensitivity.
SENSITIVITY_REPEATS = 3
LAMBDA_GRID = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
DELTA_GRID = [0.5, 1.0, 2.0]
SENS_BOOTSTRAPS = 5
SENS_DECOY_TRIALS = 5

# Experiment 4: representative main-text LIME/SHAP stress comparison.
BASELINE_DATASETS = DATASETS.copy()
BASELINE_LEARNER = "lgbm"  # native TreeSHAP is exact for this representative learner
BASELINE_REPEATS = 2
BASELINE_CONDITIONS = [(0.0, 0.0), (0.10, 0.50)]
BASELINE_EVAL_N = 12
BASELINE_BOOTSTRAPS = 10
BASELINE_NOISE_REPEATS = 3
LIME_NUM_SAMPLES = 500
LIME_NUM_FEATURES = 50
LIME_RIDGE = 1e-3

if QUICK_TEST:
    DATASETS = ["breast_cancer"]
    LEARNERS = ["lgbm"]
    MAX_N_PER_DATASET = {"breast_cancer": 300}
    MAX_CLONES = 10
    ADDITIONAL_REPEATS = 1
    E2E_REPEATS = 1
    ADDITIONAL_OUTLIER_LEVELS = [0.10]
    ADDITIONAL_CLONE_FRACS = [0.50]
    ADDITIONAL_BOOTSTRAPS = 2
    ADDITIONAL_CI_RESAMPLES = 300
    IRRELEVANT_DECOY_TRIALS = 2
    SYN_REPEATS = 2
    SYN_N = 300
    SYN_D = 30
    SYN_K = 5
    SYN_RHOS = [0.70]
    SYN_OUTLIER_LEVELS = [0.10]
    SENSITIVITY_REPEATS = 1
    LAMBDA_GRID = [1e-3, 1e-2]
    DELTA_GRID = [1.0]
    SENS_BOOTSTRAPS = 2
    SENS_DECOY_TRIALS = 2
    BASELINE_DATASETS = ["breast_cancer"]
    BASELINE_REPEATS = 1
    BASELINE_EVAL_N = 3
    BASELINE_BOOTSTRAPS = 2
    BASELINE_NOISE_REPEATS = 1
    LIME_NUM_SAMPLES = 80
    LIME_NUM_FEATURES = 15

print("Pipeline version:", PIPELINE_VERSION)
print("Output directory:", ADDITIONAL_OUTDIR)
print("Quick test:", QUICK_TEST)
print("Force recompute:", FORCE_RECOMPUTE)
print("Run end-to-end protocol:", RUN_END_TO_END)
print("Run LIME/SHAP baselines:", RUN_BASELINES)
print("Datasets:", DATASETS)
print("Learners:", LEARNERS)
print("Inverse-map orientation: X ≈ Y A^T")
print("Huber residual definition: row RMS input-space residual")
print("Default lambda:", DEFAULT_RIDGE_LAMBDA)
print("Default delta:", DEFAULT_HUBER_DELTA)


Pipeline version: additional_validation_inverse_map_2026-08-31_v3_s11_s16
Output directory: /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures
Quick test: False
Force recompute: True
Run end-to-end protocol: True
Run LIME/SHAP baselines: True
Datasets: ['breast_cancer', 'credit_approval', 'har']
Learners: ['lgbm', 'svm', 'mlp']
Inverse-map orientation: X ≈ Y A^T
Huber residual definition: row RMS input-space residual
Default lambda: 0.01
Default delta: 1.0


In [3]:

# %% [dependency check: genuine LightGBM and native TreeSHAP]
def ensure_lightgbm():
    try:
        from lightgbm import LGBMClassifier
        return LGBMClassifier, True, ""
    except Exception as first_error:
        if AUTO_INSTALL:
            print("Installing LightGBM...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
            try:
                from lightgbm import LGBMClassifier
                return LGBMClassifier, True, ""
            except Exception as second_error:
                return None, False, f"{first_error}; after install: {second_error}"
        return None, False, str(first_error)

LGBMClassifier, LGBM_AVAILABLE, LGBM_ERROR = ensure_lightgbm()

if not LGBM_AVAILABLE and not QUICK_TEST:
    raise RuntimeError(
        "LightGBM is required for the full experiment and native TreeSHAP comparison. "
        f"Import/install error: {LGBM_ERROR}"
    )

print("LightGBM available:", LGBM_AVAILABLE)
print("LIME fallback implementation: internal deterministic locally weighted surrogate")
print("SHAP implementation: LightGBM native TreeSHAP (pred_contrib=True)")


def ensure_lime():
    try:
        from lime.lime_tabular import LimeTabularExplainer
        return LimeTabularExplainer, True, "lime.lime_tabular.LimeTabularExplainer"
    except Exception as first_error:
        if AUTO_INSTALL:
            try:
                print("Installing LIME...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])
                from lime.lime_tabular import LimeTabularExplainer
                return LimeTabularExplainer, True, "lime.lime_tabular.LimeTabularExplainer"
            except Exception as second_error:
                return None, False, f"internal fallback; install error: {first_error}; {second_error}"
        return None, False, f"internal fallback; import error: {first_error}"

LimeTabularExplainer, LIME_PACKAGE_AVAILABLE, LIME_IMPLEMENTATION = ensure_lime()
print("LIME package available:", LIME_PACKAGE_AVAILABLE)
print("LIME implementation selected:", LIME_IMPLEMENTATION)
if RUN_BASELINES and (not QUICK_TEST) and (not LIME_PACKAGE_AVAILABLE):
    raise RuntimeError(
        "The full baseline-comparison experiment requires the standard `lime` package. "
        "Install it with `%pip install lime`, restart the runtime, and run all cells again. "
        "The internal fallback is permitted only for the smoke test."
    )


LightGBM available: True
LIME fallback implementation: internal deterministic locally weighted surrogate
SHAP implementation: LightGBM native TreeSHAP (pred_contrib=True)
LIME package available: True
LIME implementation selected: lime.lime_tabular.LimeTabularExplainer



## Shared helpers, data loading, stress generation, and inverse-map estimators

The same processed public datasets and three black-box learner families are retained. The central additional real-data experiment deliberately keeps the black box fixed and corrupts only the explanation-fitting matrix. This isolates the incremental value of the Huber term over RidgeAIME.


In [4]:

# %% [file/table helpers and versioned cache]

def flatten_columns(df):
    """Return a copy with stable one-line column names for CSV/LaTeX export."""
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        names = []
        for tup in out.columns.to_flat_index():
            parts = [str(x) for x in tup if str(x) not in {"", "None"}]
            names.append("__".join(parts))
        out.columns = names
    else:
        out.columns = [str(c) for c in out.columns]
    return out

def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = df.copy()
    if "pipeline_version" not in out.columns:
        out.insert(0, "pipeline_version", PIPELINE_VERSION)
    out.to_csv(path, index=False)
    print("[csv]", path)
    return path

def load_versioned_csv(path):
    path = Path(path)
    if not path.exists() or FORCE_RECOMPUTE:
        return None
    df = pd.read_csv(path)
    if "pipeline_version" not in df.columns or not (df["pipeline_version"] == PIPELINE_VERSION).all():
        print("[cache ignored: version mismatch]", path)
        return None
    print("[cache]", path)
    return df

def save_table_bundle(df, stem, caption, label, index=False):
    csv_path = ADDITIONAL_DATADIR / f"{stem}.csv"
    tex_path = ADDITIONAL_TABLEDIR / f"{stem}.tex"
    flat = flatten_columns(df)
    save_csv(flat, csv_path)
    tex_df = flat.drop(columns=["pipeline_version"], errors="ignore")
    tex = tex_df.to_latex(
        index=index, escape=False, caption=caption, label=label, position="t"
    )
    tex_path.write_text(tex, encoding="utf-8")
    print("[tex]", tex_path)
    return csv_path, tex_path

def file_sha256(path, block_size=2**20):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def make_unique(names):
    seen, out = {}, []
    for name in map(str, names):
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}__dup{seen[name]}")
    return out

def cosine_safe(a, b):
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if denom < 1e-12 else float(np.dot(a, b) / denom)

def spearman_safe(a, b):
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(stats.spearmanr(a, b).correlation)

def topk_set(scores, k=10):
    scores = np.asarray(scores, float)
    if scores.size == 0:
        return set()
    k = min(int(k), scores.size)
    return set(np.argpartition(-scores, k - 1)[:k].tolist())

def topk_jaccard(a, b, k=10):
    A, B = topk_set(a, k), topk_set(b, k)
    return np.nan if not (A | B) else len(A & B) / len(A | B)

def global_strength(A):
    return np.linalg.norm(np.asarray(A, float), axis=1)

def cluster_bootstrap_ci(cluster_values, n_resamples=10000, confidence=0.95, seed=42):
    values = np.asarray(cluster_values, float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return dict(mean=np.nan, median=np.nan, ci_low=np.nan, ci_high=np.nan, n_clusters=0)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    boot = values[idx].mean(axis=1)
    alpha = (1 - confidence) / 2
    return dict(
        mean=float(values.mean()),
        median=float(np.median(values)),
        ci_low=float(np.quantile(boot, alpha)),
        ci_high=float(np.quantile(boot, 1 - alpha)),
        n_clusters=int(len(values)),
    )

def wilcoxon_two_sided(values):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2 or np.allclose(values, 0):
        return np.nan, 1.0
    stat, p = stats.wilcoxon(values, alternative="two-sided", zero_method="wilcox")
    return float(stat), float(p)


In [5]:

# %% [data loading]
def _download_bytes(urls, timeout=180, retries=3, sleep=2):
    if isinstance(urls, str):
        urls = [urls]
    last = None
    for url in urls:
        for _ in range(retries):
            try:
                with urllib.request.urlopen(url, timeout=timeout) as r:
                    return r.read()
            except Exception as e:
                last = e
                time.sleep(sleep)
    raise last if last is not None else RuntimeError("download failed")

def load_breast_cancer_dataset():
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=make_unique(data.feature_names))
    y = pd.Series(data.target.astype(int), name="target")
    meta = {
        "dataset": "breast_cancer",
        "raw_missing_cells": int(X.isna().sum().sum()),
        "source": "sklearn/UCI WDBC",
    }
    return X, y, meta

def load_credit_raw():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/credit-screening/crx.data"
    raw = pd.read_csv(url, header=None, na_values="?")
    raw.columns = [f"A{i+1}" for i in range(raw.shape[1] - 1)] + ["target"]
    return raw

def process_credit(protocol="impute"):
    raw = load_credit_raw()
    proc = raw.dropna().copy() if protocol == "complete_case" else raw.copy()
    y = proc["target"].map({"+": 1, "-": 0}).astype(int)
    X = proc.drop(columns=["target"]).copy()
    cat_cols = [c for c in X.columns if X[c].dtype == object]
    num_cols = [c for c in X.columns if c not in cat_cols]
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
        if protocol == "impute":
            X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype("object")
        if protocol == "impute":
            X[c] = X[c].fillna("Unknown")
    X = pd.get_dummies(X, drop_first=True)
    X.columns = make_unique(X.columns)
    return X.astype(float), y.reset_index(drop=True), raw

def load_credit_approval():
    X, y, raw = process_credit("impute")
    return X, y, {
        "dataset": "credit_approval",
        "raw_missing_cells": int(raw.drop(columns=["target"]).isna().sum().sum()),
        "source": "UCI Australian Credit Approval",
    }

def load_har():
    mirrors = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
    ]
    by = _download_bytes(mirrors)
    zf = zipfile.ZipFile(io.BytesIO(by))
    root = "UCI HAR Dataset/"
    with zf.open(root + "features.txt") as f:
        feats = pd.read_csv(f, sep=r"\s+", header=None, names=["idx", "name"])
    names = make_unique(feats["name"].astype(str))
    with zf.open(root + "train/X_train.txt") as f:
        Xtr = np.loadtxt(f)
    with zf.open(root + "test/X_test.txt") as f:
        Xte = np.loadtxt(f)
    with zf.open(root + "train/y_train.txt") as f:
        ytr = np.loadtxt(f).astype(int).ravel()
    with zf.open(root + "test/y_test.txt") as f:
        yte = np.loadtxt(f).astype(int).ravel()
    X = pd.DataFrame(np.vstack([Xtr, Xte]), columns=names)
    y = pd.Series(np.hstack([ytr, yte]) - 1, name="target")
    return X, y, {
        "dataset": "har",
        "raw_missing_cells": int(X.isna().sum().sum()),
        "source": "UCI HAR",
    }

def load_dataset(name):
    if name == "breast_cancer":
        return load_breast_cancer_dataset()
    if name == "credit_approval":
        return load_credit_approval()
    if name == "har":
        return load_har()
    raise ValueError(name)

def stratified_cap(X, y, max_n=None, seed=42):
    X, y = np.asarray(X, float), np.asarray(y, int)
    if max_n is None or len(y) <= max_n:
        return X, y
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=max_n, random_state=seed)
    idx, _ = next(splitter.split(X, y))
    return X[idx], y[idx]

def standardized_dataset(name, seed=42):
    Xdf, ys, meta = load_dataset(name)
    X = StandardScaler().fit_transform(Xdf.to_numpy(float))
    y = ys.to_numpy(int)
    X, y = stratified_cap(X, y, MAX_N_PER_DATASET.get(name), seed)
    return X, y, meta


In [6]:
# %% [stress generation with observable clean counterpart and true irrelevant decoys]
def inject_outliers_with_mask(X, rate=0.10, scale=8.0, feature_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float).copy()
    n, d = X.shape
    row_mask = np.zeros(n, dtype=bool)
    cell_mask = np.zeros((n, d), dtype=bool)
    if rate <= 0:
        return X, row_mask, cell_mask
    m = max(1, int(round(rate * n)))
    rows = rng.choice(n, size=m, replace=False)
    row_mask[rows] = True
    q = max(1, int(round(feature_frac * d)))
    for i in rows:
        cols = rng.choice(d, size=q, replace=False)
        cell_mask[i, cols] = True
        perturb = np.clip(rng.standard_t(df=2, size=q) * scale, -5 * scale, 5 * scale)
        X[i, cols] += perturb
    return X, row_mask, cell_mask

def make_clone_spec(X_clean, clone_frac=0.50, clone_noise=0.01, max_clones=100, seed=42):
    X_clean = np.asarray(X_clean, float)
    n, d = X_clean.shape
    if clone_frac <= 0:
        return {"n_clones": 0, "source": np.array([], int), "aux": np.array([], int),
                "noise": np.empty((n, 0)), "clone_noise": clone_noise}
    rng = np.random.default_rng(seed)
    q = max(1, min(int(round(clone_frac * d)), max_clones))
    source = rng.choice(d, size=q, replace=True)
    aux = rng.choice(d, size=q, replace=True)
    noise = np.empty((n, q))
    for j, s in enumerate(source):
        noise[:, j] = rng.normal(0, clone_noise * (np.std(X_clean[:, s]) + 1e-12), size=n)
    return {"n_clones": q, "source": source, "aux": aux, "noise": noise, "clone_noise": clone_noise}

def apply_clone_spec(X, spec):
    X = np.asarray(X, float)
    if spec["n_clones"] == 0:
        return X.copy()
    clones = np.column_stack([
        0.95 * X[:, s] + 0.05 * X[:, t] + spec["noise"][:, j]
        for j, (s, t) in enumerate(zip(spec["source"], spec["aux"]))
    ])
    return np.column_stack([X, clones])

def build_stress_pair(X_clean, outlier_rate, clone_frac, seed):
    X_out, row_mask, cell_mask = inject_outliers_with_mask(
        X_clean, rate=outlier_rate, seed=seed
    )
    spec = make_clone_spec(
        X_clean, clone_frac=clone_frac, max_clones=MAX_CLONES, seed=seed + 1000
    )
    X_clean_aug = apply_clone_spec(X_clean, spec)
    X_stress_aug = apply_clone_spec(X_out, spec)
    d0 = X_clean.shape[1]
    clone_idx = list(range(d0, X_clean_aug.shape[1]))
    info = {
        "outlier_rate": float(outlier_rate),
        "clone_frac": float(clone_frac),
        "n_outlier_rows": int(row_mask.sum()),
        "n_clones": int(spec["n_clones"]),
        "d_original": int(d0),
        "d_total": int(X_clean_aug.shape[1]),
    }
    return X_clean_aug, X_stress_aug, row_mask, cell_mask, clone_idx, info

def append_irrelevant_decoys(X, n_decoys=3, seed=42):
    """Append distribution-matched but target-irrelevant columns.

    Each decoy is a row permutation of an existing column plus tiny noise. The marginal
    distribution is retained while its pairing with Y is broken. This is distinct from
    the near-collinear clone features used to induce multicollinearity.
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float)
    n, d = X.shape
    decoys = []
    for _ in range(n_decoys):
        src = int(rng.integers(0, d))
        col = rng.permutation(X[:, src]).copy()
        col += rng.normal(0, 0.01 * (np.std(col) + 1e-12), size=n)
        decoys.append(col)
    Xd = np.column_stack([X, np.column_stack(decoys)])
    decoy_idx = list(range(d, d + n_decoys))
    return Xd, decoy_idx


In [7]:

# %% [black-box learners]
def make_learner(kind, seed=42):
    if kind == "lgbm":
        if not LGBM_AVAILABLE:
            if QUICK_TEST:
                return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=seed)
            raise RuntimeError("Full experiment requires LightGBM.")
        return LGBMClassifier(
            n_estimators=250, learning_rate=0.05, max_depth=10,
            subsample=0.9, colsample_bytree=0.9, random_state=seed,
            verbosity=-1, n_jobs=-1,
        )
    if kind == "svm":
        return make_pipeline(
            StandardScaler(),
            SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=seed),
        )
    if kind == "mlp":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(100, 50), activation="relu",
                learning_rate_init=1e-3, alpha=1e-4, max_iter=250,
                early_stopping=True, random_state=seed,
            ),
        )
    raise ValueError(kind)

def model_classes(model):
    if hasattr(model, "classes_"):
        return np.asarray(model.classes_)
    if hasattr(model, "named_steps"):
        last = list(model.named_steps.values())[-1]
        if hasattr(last, "classes_"):
            return np.asarray(last.classes_)
    return None

def fit_clean_blackbox(X, y, learner, seed):
    idx = np.arange(len(y))
    train_idx, exp_idx = train_test_split(
        idx, test_size=0.30, random_state=seed, stratify=y
    )
    model = make_learner(learner, seed)
    t0 = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_time = time.perf_counter() - t0
    Y_exp = model.predict_proba(X[exp_idx])
    pred_idx = np.argmax(Y_exp, axis=1)
    classes = model_classes(model)
    pred = classes[pred_idx] if classes is not None else pred_idx
    acc = accuracy_score(y[exp_idx], pred)
    try:
        ll = log_loss(y[exp_idx], Y_exp, labels=np.unique(y))
    except Exception:
        ll = np.nan
    return model, train_idx, exp_idx, Y_exp, {
        "model_fit_time_sec": fit_time,
        "model_test_accuracy": float(acc),
        "model_test_log_loss": float(ll),
    }

def split_explanation_rows(y_exp, seed, fit_fraction=0.70):
    idx = np.arange(len(y_exp))
    fit_idx, val_idx = train_test_split(
        idx, train_size=fit_fraction, random_state=seed, stratify=y_exp
    )
    return np.asarray(fit_idx), np.asarray(val_idx)


## Equation-consistent AIME-family implementation

For every real and synthetic experiment below, \(A\in\mathbb{R}^{d\times C}\) is estimated from \(X\approx YA^\top\). Huber weights are **row weights** based on input-space reconstruction residuals. The RMS normalization of the row residual is recorded in every output table.

The solver uses an explicit SVD-based linear-system solution. For \(\lambda=0\), numerically null singular directions are handled by the Moore–Penrose rule. For \(\lambda>0\), ridge changes the spectrum before the SVD solve; it is therefore not equivalent to simply invoking a pseudoinverse or applying a hard truncated-SVD cutoff.


In [8]:
# %% [inverse-map solver, decoy diagnostics, and stability]
AIME_METHODS = ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]

def huber_weights(residual, delta=1.0):
    residual = np.asarray(residual, float)
    w = np.ones_like(residual)
    mask = residual > delta
    w[mask] = delta / np.maximum(residual[mask], 1e-12)
    return np.clip(w, 1e-8, 1.0)

def symmetric_condition(G):
    G = np.asarray(G, float)
    evals = np.linalg.eigvalsh((G + G.T) / 2)
    lam_min, lam_max = float(evals.min()), float(evals.max())
    cond = float(lam_max / max(lam_min, 1e-12))
    return cond, lam_min, lam_max

def svd_solve(G, rhs, rcond=1e-12):
    """SVD-based solve used for all inverse-map normal systems."""
    U, s, Vt = np.linalg.svd(np.asarray(G, float), full_matrices=False)
    tol = rcond * max(float(s.max()), 1e-300)
    inv = np.where(s > tol, 1.0 / s, 0.0)
    return (Vt.T * inv) @ (U.T @ np.asarray(rhs, float)), s, tol

def fit_inverse_map(
    X, Y, method="AIME", delta=1.0, ridge_lambda=1e-2,
    residual_mode="rms", max_iter=100, tol=1e-8
):
    """Fit A in X ≈ Y A^T using the manuscript equations."""
    X, Y = np.asarray(X, float), np.asarray(Y, float)
    if X.ndim != 2 or Y.ndim != 2 or len(X) != len(Y):
        raise ValueError(f"Expected X(n,d) and Y(n,C); got {X.shape} and {Y.shape}")
    n, d = X.shape
    C = Y.shape[1]
    use_huber = method in {"HuberAIME", "HuberRidgeAIME"}
    use_ridge = method in {"RidgeAIME", "HuberRidgeAIME"}
    lam = float(ridge_lambda if use_ridge else 0.0)

    w = np.ones(n)
    B = np.zeros((C, d))  # B = A^T
    converged = not use_huber
    n_iter = 1

    for it in range(max_iter if use_huber else 1):
        G = Y.T @ (Y * w[:, None])
        G_reg = G + lam * np.eye(C)
        rhs = Y.T @ (X * w[:, None])
        B_new, _, _ = svd_solve(G_reg, rhs)
        residual_norm = np.linalg.norm(X - Y @ B_new, axis=1)
        residual_score = residual_norm / np.sqrt(max(1, d)) if residual_mode == "rms" else residual_norm

        if not use_huber:
            B = B_new
            break

        w_new = huber_weights(residual_score, delta)
        n_iter = it + 1
        coef_change = np.linalg.norm(B_new - B) / (np.linalg.norm(B) + 1e-12)
        weight_change = np.max(np.abs(w_new - w))
        B, w = B_new, w_new
        if coef_change <= tol and weight_change <= np.sqrt(tol):
            converged = True
            break
    else:
        converged = False

    # Final SVD solve at the final IRLS weights, so B and the reported system agree exactly.
    G = Y.T @ (Y * w[:, None])
    G_reg = G + lam * np.eye(C)
    rhs = Y.T @ (X * w[:, None])
    B, system_singular_values, solver_tol = svd_solve(G_reg, rhs)
    residual_norm = np.linalg.norm(X - Y @ B, axis=1)
    residual_score = residual_norm / np.sqrt(max(1, d)) if residual_mode == "rms" else residual_norm

    cond_raw, min_raw, max_raw = symmetric_condition(G)
    cond_reg, min_reg, max_reg = symmetric_condition(G_reg)
    A = B.T
    if A.shape != (d, C):
        raise AssertionError(f"Inverse orientation failure: A has shape {A.shape}, expected {(d,C)}")

    diag = {
        "orientation": "X ~= Y A^T",
        "solver": "SVD normal-system solve",
        "cond_raw": cond_raw,
        "cond_reg": cond_reg,
        "lambda_min_raw": min_raw,
        "lambda_min_reg": min_reg,
        "lambda_max_raw": max_raw,
        "lambda_max_reg": max_reg,
        "system_min_singular_value": float(system_singular_values.min()),
        "solver_svd_tolerance": float(solver_tol),
        "coef_fro_norm": float(np.linalg.norm(A, "fro")),
        "mean_iter": int(n_iter),
        "converged": bool(converged),
        "mean_huber_weight": float(w.mean()),
        "frac_downweighted": float(np.mean(w < 0.999)),
        "ridge_lambda": lam,
        "huber_delta": float(delta),
        "residual_mode": residual_mode,
        "effective_delta_on_l2_norm": float(delta * np.sqrt(d)) if residual_mode == "rms" else float(delta),
    }
    state = {
        "A": A, "B": B, "weights": w, "residual_score": residual_score,
        "method": method, "ridge_lambda": lam, "huber_delta": float(delta),
        "residual_mode": residual_mode,
    }
    return A, diag, state

def inverse_reconstruction_metrics(state, Y, X_target, prefix=""):
    X_target = np.asarray(X_target, float)
    X_hat = np.asarray(Y, float) @ state["B"]
    err = X_target - X_hat
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    denom = float(np.sum((X_target - X_target.mean(axis=0, keepdims=True)) ** 2))
    r2 = float(1 - np.sum(err ** 2) / (denom + 1e-12))
    cos = cosine_safe(X_target, X_hat)
    return {
        f"{prefix}reconstruction_mse": mse,
        f"{prefix}reconstruction_rmse": rmse,
        f"{prefix}reconstruction_r2": r2,
        f"{prefix}reconstruction_cosine": cos,
    }

def operator_recovery_metrics(A, A_reference, prefix="operator_"):
    g, g_ref = global_strength(A), global_strength(A_reference)
    return {
        f"{prefix}cosine_flat": cosine_safe(A, A_reference),
        f"{prefix}spearman_global": spearman_safe(g, g_ref),
        f"{prefix}topk_jaccard": topk_jaccard(g, g_ref, TOP_K),
    }

def subset_mass_metrics(A, feature_idx, prefix):
    g = global_strength(A)
    if not feature_idx:
        return {f"{prefix}_mass_ratio": 0.0, f"{prefix}_topk_infiltration": 0.0}
    mass = float(g[feature_idx].sum() / (g.sum() + 1e-12))
    top = topk_set(g, TOP_K)
    infiltration = float(sum(i in top for i in feature_idx) / len(feature_idx))
    return {f"{prefix}_mass_ratio": mass, f"{prefix}_topk_infiltration": infiltration}

def outlier_weight_metrics(state, known_outlier_mask):
    mask = np.asarray(known_outlier_mask, bool)
    weights = np.asarray(state["weights"], float)
    if len(mask) != len(weights) or mask.sum() == 0 or (~mask).sum() == 0:
        return {
            "outlier_weight_ap": np.nan,
            "mean_weight_outlier_rows": np.nan,
            "mean_weight_nonoutlier_rows": np.nan,
        }
    return {
        "outlier_weight_ap": float(average_precision_score(mask.astype(int), 1 - weights)),
        "mean_weight_outlier_rows": float(weights[mask].mean()),
        "mean_weight_nonoutlier_rows": float(weights[~mask].mean()),
    }

def bootstrap_inverse_stability(X, Y, method, delta, lam, A0=None, B=10, seed=42):
    rng = np.random.default_rng(seed)
    if A0 is None:
        A0, _, _ = fit_inverse_map(X, Y, method, delta, lam, RESIDUAL_SCALE_MODE)
    vals = []
    n = len(X)
    for _ in range(B):
        idx = rng.choice(n, size=n, replace=True)
        Ab, _, _ = fit_inverse_map(X[idx], Y[idx], method, delta, lam, RESIDUAL_SCALE_MODE)
        vals.append({
            "bootstrap_cosine_flat": cosine_safe(A0, Ab),
            "bootstrap_spearman_global": spearman_safe(global_strength(A0), global_strength(Ab)),
            "bootstrap_topk_jaccard": topk_jaccard(global_strength(A0), global_strength(Ab), TOP_K),
        })
    return pd.DataFrame(vals).mean().to_dict()

def irrelevant_decoy_metrics(X, Y, method, delta, lam, trials=5, seed=42):
    rows = []
    for t in range(trials):
        Xd, decoy_idx = append_irrelevant_decoys(X, IRRELEVANT_DECOY_COUNT, seed + 1009 * t)
        Ad, _, _ = fit_inverse_map(Xd, Y, method, delta, lam, RESIDUAL_SCALE_MODE)
        rows.append(subset_mass_metrics(Ad, decoy_idx, "irrelevant_decoy"))
    return pd.DataFrame(rows).mean().to_dict()


In [9]:

# %% [dataset diagnostics for additional validation]
def effective_rank(s):
    s = np.asarray(s, float)
    s = s[s > 1e-12]
    if not len(s):
        return 0.0
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p + 1e-15)).sum()))

def dataset_diagnostics(X):
    Xs = StandardScaler().fit_transform(np.asarray(X, float))
    n, d = Xs.shape
    s = np.linalg.svd(Xs, compute_uv=False, full_matrices=False)
    cond_xtx = float((s[0] / max(s[-1], 1e-12)) ** 2)
    C = np.nan_to_num(np.corrcoef(Xs, rowvar=False), nan=0.0)
    iu = np.triu_indices_from(C, 1)
    abs_corr = np.abs(C[iu])
    med = np.median(Xs, axis=0)
    mad = np.median(np.abs(Xs - med), axis=0)
    rz = 0.6745 * (Xs - med) / np.where(mad < 1e-12, 1.0, mad)
    kurt = stats.kurtosis(Xs, axis=0, fisher=True, bias=False, nan_policy="omit")
    return {
        "n": n, "d": d,
        "matrix_rank": int(np.linalg.matrix_rank(Xs)),
        "effective_rank_ratio": effective_rank(s) / max(1, d),
        "near_zero_singular_values": int(np.sum(s <= 1e-10 * max(s[0], 1e-12))),
        "log10_condition_XtX": float(np.log10(max(cond_xtx, 1e-300))),
        "median_abs_correlation": float(np.median(abs_corr)),
        "frac_abs_corr_gt_0.50": float(np.mean(abs_corr > 0.50)),
        "frac_abs_corr_gt_0.70": float(np.mean(abs_corr > 0.70)),
        "frac_abs_corr_gt_0.90": float(np.mean(abs_corr > 0.90)),
        "robust_outlier_cell_rate": float(np.mean(np.abs(rz) > 3.5)),
        "median_excess_kurtosis": float(np.nanmedian(kurt)),
        "q90_excess_kurtosis": float(np.nanquantile(kurt, 0.90)),
        "frac_features_excess_kurtosis_gt_10": float(np.nanmean(kurt > 10)),
    }

def run_dataset_diagnostics():
    out = ADDITIONAL_DATADIR / "additional_validation_dataset_diagnostics.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    for dataset in DATASETS:
        Xdf, y, meta = load_dataset(dataset)
        rows.append({
            "dataset": dataset,
            "raw_missing_cells": meta["raw_missing_cells"],
            **dataset_diagnostics(Xdf.to_numpy(float)),
        })
    df = pd.DataFrame(rows)
    save_table_bundle(
        df, "table_additional_validation_dataset_diagnostics",
        "Extended dataset diagnostics. HAR's high cell-level robust-outlier rate is interpreted together with its feature kurtosis; Australian Credit's ill-conditioning is interpreted using its rank and spectrum rather than only pairwise correlations.",
        "tab:additional_validation_dataset_diagnostics",
    )
    return df

ADDITIONAL_DATASET_DIAGNOSTICS = run_dataset_diagnostics()
display(ADDITIONAL_DATASET_DIAGNOSTICS)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_dataset_diagnostics.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_dataset_diagnostics.tex


,dataset,raw_missing_cells,n,d,matrix_rank,effective_rank_ratio,near_zero_singular_values,log10_condition_XtX,median_abs_correlation,frac_abs_corr_gt_0.50,frac_abs_corr_gt_0.70,frac_abs_corr_gt_0.90,robust_outlier_cell_rate,median_excess_kurtosis,q90_excess_kurtosis,frac_features_excess_kurtosis_gt_10
0,breast_cancer,0,569,30,30,0.558777,0,4.999253,0.345007,0.333333,0.160920,0.048276,0.029818,3.022590,21.889799,0.200000
1,credit_approval,67,690,42,38,0.801707,4,27.472007,0.040543,0.019744,0.011614,0.010453,0.017219,8.049270,210.076450,0.452381
2,har,0,10299,561,540,0.366865,21,30.467093,0.379293,0.413032,0.230768,0.051522,0.234115,0.586898,36.072109,0.244207


## Experiment 1 — HRA versus RidgeAIME under injected outliers

This comparison isolates the value added by the robust loss under contamination.

### Protocol A: fixed-black-box contamination

A clean black box is trained once. Its output matrix \(Y\) is fixed while outliers are injected into the response side of the inverse explanation problem. This directly isolates whether Huber row weighting contributes anything beyond ridge regularization.

### Protocol B: original end-to-end stress design

The black box is trained and evaluated on the stressed feature matrix, matching the controlled-stress protocol described in the manuscript. Reporting both protocols prevents the additional experiment from answering a different question than the submitted study.

For both protocols the notebook reports:

- clean-target or observed-target inverse reconstruction (**not labeled as ground-truth faithfulness**),
- recovery of a clean-reference inverse operator,
- bootstrap stability,
- mass assigned to correlated clones,
- mass assigned to genuinely irrelevant permutation decoys,
- Huber weights against the known injected-outlier row mask,
- condition-cluster bootstrap 95% confidence intervals.


The final manuscript-ready multi-panel figures are generated directly by this notebook. Display-only horizontal offsets are documented in the figure code and do not change experimental values.


In [10]:
# %% [Experiment 1: direct and end-to-end HRA-vs-RidgeAIME comparisons]
def _collect_inverse_method_row(
    *, experiment_mode, dataset, learner, rep, method, stress_info, model_info,
    X_fit, Y_fit, X_val_observed, Y_val_observed,
    X_val_clean, Y_val_clean, A_ref, clone_idx, outlier_mask_fit, seed
):
    t0 = time.perf_counter()
    A, diag, state = fit_inverse_map(
        X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
        RESIDUAL_SCALE_MODE,
    )
    explain_time = time.perf_counter() - t0
    clean_rec = inverse_reconstruction_metrics(state, Y_val_clean, X_val_clean, "clean_target_")
    observed_rec = inverse_reconstruction_metrics(state, Y_val_observed, X_val_observed, "observed_target_")
    op = operator_recovery_metrics(A, A_ref)
    clone = subset_mass_metrics(A, clone_idx, "clone")
    decoy = irrelevant_decoy_metrics(
        X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
        IRRELEVANT_DECOY_TRIALS, seed + 61,
    )
    outlier_diag = outlier_weight_metrics(state, outlier_mask_fit)
    boot = bootstrap_inverse_stability(
        X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
        A0=A, B=ADDITIONAL_BOOTSTRAPS, seed=seed + 29,
    )
    return {
        "experiment_mode": experiment_mode,
        "dataset": dataset, "learner": learner, "repeat": rep,
        "method": method, **stress_info, **model_info,
        "n_explanation_fit": len(X_fit), "n_explanation_val": len(X_val_observed),
        "explain_time_sec": explain_time,
        **diag, **clean_rec, **observed_rec, **op, **clone, **decoy,
        **outlier_diag, **boot, "error": "",
    }

def run_fixed_black_box_contamination():
    out = ADDITIONAL_DATADIR / "additional_validation_hra_vs_ridge_fixed_black_box_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    total = len(DATASETS) * len(LEARNERS) * ADDITIONAL_REPEATS
    job = 0
    for dataset in DATASETS:
        X, y, _ = standardized_dataset(dataset, ADDITIONAL_SEED)
        for rep in range(ADDITIONAL_REPEATS):
            for learner in LEARNERS:
                job += 1
                seed_model = ADDITIONAL_SEED + rep * 10000 + 100 * (LEARNERS.index(learner) + 1)
                print(f"[fixed-black-box {job}/{total}] {dataset} {learner} rep={rep}")
                model, _, exp_idx, Y_exp, model_info = fit_clean_blackbox(X, y, learner, seed_model)
                X_exp_clean, y_exp = X[exp_idx], y[exp_idx]
                for outlier_rate in ADDITIONAL_OUTLIER_LEVELS:
                    for clone_frac in ADDITIONAL_CLONE_FRACS:
                        seed = seed_model + int(outlier_rate * 1000) + int(clone_frac * 100)
                        X_clean_aug, X_stress_aug, out_mask, _, clone_idx, stress_info = build_stress_pair(
                            X_exp_clean, outlier_rate, clone_frac, seed
                        )
                        fit_idx, val_idx = split_explanation_rows(y_exp, seed + 17)
                        A_ref, _, _ = fit_inverse_map(
                            X_clean_aug[fit_idx], Y_exp[fit_idx], "RidgeAIME",
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
                        )
                        for method in ["RidgeAIME", "HuberRidgeAIME"]:
                            rows.append(_collect_inverse_method_row(
                                experiment_mode="fixed_black_box_contamination",
                                dataset=dataset, learner=learner, rep=rep, method=method,
                                stress_info=stress_info, model_info=model_info,
                                X_fit=X_stress_aug[fit_idx], Y_fit=Y_exp[fit_idx],
                                X_val_observed=X_stress_aug[val_idx], Y_val_observed=Y_exp[val_idx],
                                X_val_clean=X_clean_aug[val_idx], Y_val_clean=Y_exp[val_idx],
                                A_ref=A_ref, clone_idx=clone_idx,
                                outlier_mask_fit=out_mask[fit_idx], seed=seed,
                            ))
    df = pd.DataFrame(rows)
    df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
    df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
    save_csv(df, out)
    return df

def run_end_to_end_original_stress():
    out = ADDITIONAL_DATADIR / "additional_validation_hra_vs_ridge_end_to_end_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_END_TO_END:
        return pd.DataFrame()
    rows = []
    total = len(DATASETS) * len(LEARNERS) * E2E_REPEATS * len(ADDITIONAL_OUTLIER_LEVELS) * len(ADDITIONAL_CLONE_FRACS)
    job = 0
    for dataset in DATASETS:
        X, y, _ = standardized_dataset(dataset, ADDITIONAL_SEED)
        for rep in range(E2E_REPEATS):
            for outlier_rate in ADDITIONAL_OUTLIER_LEVELS:
                for clone_frac in ADDITIONAL_CLONE_FRACS:
                    seed = ADDITIONAL_SEED + rep * 10000 + int(outlier_rate * 1000) + int(clone_frac * 100)
                    X_clean_aug, X_stress_aug, out_mask, _, clone_idx, stress_info = build_stress_pair(
                        X, outlier_rate, clone_frac, seed
                    )
                    for learner in LEARNERS:
                        job += 1
                        print(f"[end-to-end {job}/{total}] {dataset} {learner} rep={rep} pi={outlier_rate} gamma={clone_frac}")
                        train_idx, test_idx = train_test_split(
                            np.arange(len(y)), test_size=0.30, random_state=seed + 100 * (LEARNERS.index(learner)+1), stratify=y
                        )
                        model = make_learner(learner, seed)
                        t0 = time.perf_counter()
                        model.fit(X_stress_aug[train_idx], y[train_idx])
                        model_fit_time = time.perf_counter() - t0
                        Y_stress = model.predict_proba(X_stress_aug[test_idx])
                        Y_clean = model.predict_proba(X_clean_aug[test_idx])
                        classes = model_classes(model)
                        pred_idx = np.argmax(Y_stress, axis=1)
                        pred = classes[pred_idx] if classes is not None else pred_idx
                        model_info = {
                            "model_fit_time_sec": model_fit_time,
                            "model_test_accuracy": float(accuracy_score(y[test_idx], pred)),
                            "model_test_log_loss": float(log_loss(y[test_idx], Y_stress, labels=np.unique(y))),
                        }
                        y_test = y[test_idx]
                        fit_idx, val_idx = split_explanation_rows(y_test, seed + 17)
                        X_clean_test, X_stress_test = X_clean_aug[test_idx], X_stress_aug[test_idx]
                        out_mask_test = out_mask[test_idx]
                        A_ref, _, _ = fit_inverse_map(
                            X_clean_test[fit_idx], Y_clean[fit_idx], "RidgeAIME",
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
                        )
                        for method in ["RidgeAIME", "HuberRidgeAIME"]:
                            rows.append(_collect_inverse_method_row(
                                experiment_mode="end_to_end_original_stress",
                                dataset=dataset, learner=learner, rep=rep, method=method,
                                stress_info=stress_info, model_info=model_info,
                                X_fit=X_stress_test[fit_idx], Y_fit=Y_stress[fit_idx],
                                X_val_observed=X_stress_test[val_idx], Y_val_observed=Y_stress[val_idx],
                                X_val_clean=X_clean_test[val_idx], Y_val_clean=Y_clean[val_idx],
                                A_ref=A_ref, clone_idx=clone_idx,
                                outlier_mask_fit=out_mask_test[fit_idx], seed=seed,
                            ))
    df = pd.DataFrame(rows)
    if len(df):
        df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
        df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
        save_csv(df, out)
    return df

REAL_METRICS = [
    ("clean_target_reconstruction_r2", "higher is better", "clean-target inverse reconstruction"),
    ("clean_target_reconstruction_cosine", "higher is better", "clean-target inverse reconstruction"),
    ("operator_cosine_flat", "higher is better", "clean-reference operator recovery"),
    ("operator_spearman_global", "higher is better", "clean-reference operator recovery"),
    ("operator_topk_jaccard", "higher is better", "clean-reference operator recovery"),
    ("bootstrap_cosine_flat", "higher is better", "self-consistency/stability"),
    ("irrelevant_decoy_mass_ratio", "lower is better", "irrelevant-decoy attribution"),
    ("irrelevant_decoy_topk_infiltration", "lower is better", "irrelevant-decoy attribution"),
    ("clone_mass_ratio", "descriptive", "correlated-clone attribution redistribution"),
    ("clone_topk_infiltration", "descriptive", "correlated-clone attribution redistribution"),
    ("log10_cond_reg", "lower is better", "conditioning diagnostic"),
    ("log10_coef_norm", "diagnostic only", "regularization diagnostic"),
]

def paired_cluster_summary(df):
    if df.empty:
        return pd.DataFrame()
    pair_key = ["experiment_mode", "dataset", "learner", "outlier_rate", "clone_frac", "repeat"]
    cluster_key = ["experiment_mode", "dataset", "learner", "outlier_rate", "clone_frac"]
    rows = []
    for mode, mode_df in df.groupby("experiment_mode"):
        subsets = [("all_outlier_present", mode_df)]
        for rate in sorted(mode_df["outlier_rate"].unique()):
            subsets.append((f"outlier_rate={rate:.2f}", mode_df[np.isclose(mode_df["outlier_rate"], rate)]))
        subsets.append(("joint_high_stress", mode_df[
            np.isclose(mode_df["outlier_rate"], max(ADDITIONAL_OUTLIER_LEVELS)) &
            np.isclose(mode_df["clone_frac"], max(ADDITIONAL_CLONE_FRACS))
        ]))
        for subset_name, sub in subsets:
            for metric, direction, role in REAL_METRICS:
                piv = sub.pivot_table(index=pair_key, columns="method", values=metric, aggfunc="mean")
                if not {"RidgeAIME", "HuberRidgeAIME"}.issubset(piv.columns):
                    continue
                piv = piv[["RidgeAIME", "HuberRidgeAIME"]].dropna().reset_index()
                piv["difference"] = piv["HuberRidgeAIME"] - piv["RidgeAIME"]
                cluster_diff = piv.groupby(cluster_key, dropna=False)["difference"].mean()
                ci = cluster_bootstrap_ci(cluster_diff.to_numpy(), ADDITIONAL_CI_RESAMPLES, seed=ADDITIONAL_SEED + len(rows))
                _, p = wilcoxon_two_sided(cluster_diff.to_numpy())
                rows.append({
                    "experiment_mode": mode, "subset": subset_name, "metric": metric,
                    "role": role, "direction": direction,
                    "n_repeated_pairs": len(piv), "n_condition_clusters": ci["n_clusters"],
                    "HRA_mean": float(piv["HuberRidgeAIME"].mean()),
                    "RidgeAIME_mean": float(piv["RidgeAIME"].mean()),
                    "HRA_minus_Ridge_cluster_mean": ci["mean"],
                    "CI95_low": ci["ci_low"], "CI95_high": ci["ci_high"],
                    "median_cluster_difference": ci["median"],
                    "wilcoxon_p_on_cluster_means": p,
                })
    return pd.DataFrame(rows)

def summarize_hra_weight_diagnostics(df):
    if df.empty:
        return pd.DataFrame()
    sub = df[df["method"] == "HuberRidgeAIME"].copy()
    metrics = ["frac_downweighted", "outlier_weight_ap", "mean_weight_outlier_rows", "mean_weight_nonoutlier_rows"]
    return sub.groupby(["experiment_mode", "outlier_rate", "clone_frac"])[metrics].agg(["mean", "std", "count"]).reset_index()

def cluster_ci_rate_subset(ci_df, mode, metric):
    sub = ci_df[
        (ci_df["experiment_mode"] == mode)
        & (ci_df["metric"] == metric)
        & ci_df["subset"].str.startswith("outlier_rate=")
    ].copy()
    return sub.sort_values("subset")


def plot_cluster_ci_on_axis(ax, ci_df, mode, metric, ylabel, title):
    sub = cluster_ci_rate_subset(ci_df, mode, metric)
    if sub.empty:
        ax.text(0.5, 0.5, "No results", ha="center", va="center", transform=ax.transAxes)
        return sub
    x = np.arange(len(sub))
    y = sub["HRA_minus_Ridge_cluster_mean"].to_numpy(float)
    lo = y - sub["CI95_low"].to_numpy(float)
    hi = sub["CI95_high"].to_numpy(float) - y
    ax.errorbar(x, y, yerr=np.vstack([lo, hi]), fmt="o", capsize=4, color="#1f77b4")
    ax.axhline(0, linewidth=1, color="0.35")
    ax.set_xticks(x, [s.replace("outlier_rate=", "pi=") for s in sub["subset"]])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.20, linewidth=0.6)
    return sub


def plot_cluster_ci(ci_df, mode, metric, filename, ylabel):
    fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=220)
    plot_cluster_ci_on_axis(
        ax,
        ci_df,
        mode,
        metric,
        ylabel,
        f"{mode}: HRA - RidgeAIME\ncondition-cluster bootstrap 95% CI",
    )
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)


def plot_fixed_black_box_manuscript_figure(ci_df):
    specs = [
        (
            "clean_target_reconstruction_cosine",
            "Difference in clean-target\nreconstruction cosine",
            "Clean-target reconstruction",
        ),
        (
            "operator_cosine_flat",
            "Difference in clean-reference\noperator cosine",
            "Clean-reference operator recovery",
        ),
        (
            "irrelevant_decoy_mass_ratio",
            "Difference in irrelevant-decoy mass\n(lower favors HRA)",
            "Irrelevant-decoy attribution",
        ),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), dpi=220)
    for panel_label, ax, (metric, ylabel, title) in zip("abc", axes, specs):
        plot_cluster_ci_on_axis(
            ax,
            ci_df,
            "fixed_black_box_contamination",
            metric,
            ylabel,
            title,
        )
        ax.text(
            -0.12,
            1.05,
            panel_label,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="top",
        )
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S11_fixed_black_box.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path


ADDITIONAL_FIXED_RAW = run_fixed_black_box_contamination()
ADDITIONAL_E2E_RAW = run_end_to_end_original_stress()
ADDITIONAL_REAL_RAW = pd.concat([x for x in [ADDITIONAL_FIXED_RAW, ADDITIONAL_E2E_RAW] if len(x)], ignore_index=True)
ADDITIONAL_REAL_CI = paired_cluster_summary(ADDITIONAL_REAL_RAW[ADDITIONAL_REAL_RAW["error"].fillna("") == ""])
ADDITIONAL_HRA_WEIGHT_SUMMARY = summarize_hra_weight_diagnostics(ADDITIONAL_REAL_RAW)
save_table_bundle(
    ADDITIONAL_HRA_WEIGHT_SUMMARY, "table_additional_validation_hra_weight_diagnostics",
    "HuberRidgeAIME row-weight diagnostics against the known injected-outlier row mask. These diagnose the Huber mechanism and are not quality scores.",
    "tab:additional_validation_hra_weight_diagnostics",
)
save_table_bundle(
    ADDITIONAL_REAL_CI, "table_additional_validation_hra_vs_ridge_cluster_ci",
    "Paired HRA-versus-RidgeAIME comparisons under injected outliers in both the fixed-black-box and original end-to-end stress protocols. Repeated seeds are averaged within dataset--learner--stress cells before condition-cluster bootstrap confidence intervals.",
    "tab:additional_validation_hra_vs_ridge_cluster_ci",
)
for mode, suffix in [("fixed_black_box_contamination", "fixed"), ("end_to_end_original_stress", "e2e")]:
    plot_cluster_ci(ADDITIONAL_REAL_CI, mode, "clean_target_reconstruction_cosine", f"fig_additional_validation_{suffix}_clean_reconstruction.png", "Difference in clean-target reconstruction cosine")
    plot_cluster_ci(ADDITIONAL_REAL_CI, mode, "irrelevant_decoy_mass_ratio", f"fig_additional_validation_{suffix}_irrelevant_decoy_mass.png", "Difference in irrelevant-decoy mass (lower favors HRA)")
    plot_cluster_ci(ADDITIONAL_REAL_CI, mode, "operator_cosine_flat", f"fig_additional_validation_{suffix}_operator_recovery.png", "Difference in clean-reference operator cosine")
ADDITIONAL_FIXED_BLACK_BOX_MANUSCRIPT_FIGURE = plot_fixed_black_box_manuscript_figure(ADDITIONAL_REAL_CI)
display(ADDITIONAL_REAL_CI)


[fixed-black-box 1/45] breast_cancer lgbm rep=0
[fixed-black-box 2/45] breast_cancer svm rep=0
[fixed-black-box 3/45] breast_cancer mlp rep=0
[fixed-black-box 4/45] breast_cancer lgbm rep=1
[fixed-black-box 5/45] breast_cancer svm rep=1
[fixed-black-box 6/45] breast_cancer mlp rep=1
[fixed-black-box 7/45] breast_cancer lgbm rep=2
[fixed-black-box 8/45] breast_cancer svm rep=2
[fixed-black-box 9/45] breast_cancer mlp rep=2
[fixed-black-box 10/45] breast_cancer lgbm rep=3
[fixed-black-box 11/45] breast_cancer svm rep=3
[fixed-black-box 12/45] breast_cancer mlp rep=3
[fixed-black-box 13/45] breast_cancer lgbm rep=4
[fixed-black-box 14/45] breast_cancer svm rep=4
[fixed-black-box 15/45] breast_cancer mlp rep=4
[fixed-black-box 16/45] credit_approval lgbm rep=0
[fixed-black-box 17/45] credit_approval svm rep=0
[fixed-black-box 18/45] credit_approval mlp rep=0
[fixed-black-box 19/45] credit_approval lgbm rep=1
[fixed-black-box 20/45] credit_approval svm rep=1
[fixed-black-box 21/45] credit_a

,experiment_mode,subset,metric,role,direction,n_repeated_pairs,n_condition_clusters,HRA_mean,RidgeAIME_mean,HRA_minus_Ridge_cluster_mean,CI95_low,CI95_high,median_cluster_difference,wilcoxon_p_on_cluster_means
0,end_to_end_original_stress,all_outlier_present,clean_target_reconstruction_r2,clean-target inverse reconstruction,higher is better,162,54,0.281627,0.252597,0.029030,0.023388,0.035375,0.024026,1.625698e-10
1,end_to_end_original_stress,all_outlier_present,clean_target_reconstruction_cosine,clean-target inverse reconstruction,higher is better,162,54,0.499052,0.476149,0.022903,0.018631,0.027412,0.016951,1.625698e-10
2,end_to_end_original_stress,all_outlier_present,operator_cosine_flat,clean-reference operator recovery,higher is better,162,54,0.987318,0.939263,0.048055,0.035874,0.061436,0.028015,1.625698e-10
3,end_to_end_original_stress,all_outlier_present,operator_spearman_global,clean-reference operator recovery,higher is better,162,54,0.950434,0.868740,0.081694,0.062169,0.103393,0.052350,8.077834e-10
4,end_to_end_original_stress,all_outlier_present,operator_topk_jaccard,clean-reference operator recovery,higher is better,162,54,0.691369,0.478808,0.212561,0.175314,0.249203,0.229400,4.934587e-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,fixed_black_box_contamination,joint_high_stress,irrelevant_decoy_topk_infiltration,irrelevant-decoy attribution,lower is better,45,9,0.007407,0.019259,-0.011852,-0.023704,-0.002963,0.000000,1.250000e-01
92,fixed_black_box_contamination,joint_high_stress,clone_mass_ratio,correlated-clone attribution redistribution,descriptive,45,9,0.262385,0.261986,0.000399,-0.004117,0.004121,0.000627,4.257812e-01
93,fixed_black_box_contamination,joint_high_stress,clone_topk_infiltration,correlated-clone attribution redistribution,descriptive,45,9,0.098444,0.101471,-0.003026,-0.013386,0.009926,-0.002000,4.375000e-01
94,fixed_black_box_contamination,joint_high_stress,log10_cond_reg,conditioning diagnostic,lower is better,45,9,0.271616,0.260131,0.011485,0.004272,0.017734,0.013867,2.734375e-02



## Experiment 2 — Known-ground-truth faithfulness

This experiment directly separates **faithfulness** from **stability**. A sparse \(A_{\mathrm{true}}\) is known, and clean inputs are generated by

\[
X_{\mathrm{clean}}=YA_{\mathrm{true}}^\top+\varepsilon.
\]

Outliers are then injected into \(X\). Support recovery, direction recovery, and clean held-out reconstruction are reported. A zero-operator baseline is included to demonstrate explicitly that a smaller coefficient norm is not, by itself, a quality measure.


In [11]:

# %% [Experiment 2]
def make_inverse_ground_truth(n, d, C, k, rho, outlier_rate, seed):
    rng = np.random.default_rng(seed)
    cov = (1 - rho) * np.eye(C) + rho * np.ones((C, C))
    Z = rng.normal(size=(n, C)) @ np.linalg.cholesky(cov + 1e-8 * np.eye(C)).T
    Y = softmax(Z, axis=1)
    support = np.sort(rng.choice(d, size=k, replace=False))
    A_true = np.zeros((d, C))
    magnitudes = np.linspace(2.0, 1.0, k)
    for j, feat in enumerate(support):
        direction = rng.normal(size=C)
        direction /= np.linalg.norm(direction) + 1e-12
        A_true[feat] = magnitudes[j] * direction
    X_clean = Y @ A_true.T + rng.normal(0, 0.05, size=(n, d))
    X_obs, row_mask, cell_mask = inject_outliers_with_mask(
        X_clean, rate=outlier_rate, seed=seed + 1
    )
    return X_clean, X_obs, Y, A_true, support, row_mask

def support_metrics(A, A_true, support, k):
    score = global_strength(A)
    truth_score = global_strength(A_true)
    truth = (truth_score > 0).astype(int)
    ap = float(average_precision_score(truth, score))
    pred = np.argsort(-score)[:k]
    tp = len(set(pred) & set(support))
    precision = tp / max(1, len(pred))
    recall = tp / max(1, len(support))
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    nz = np.abs(A_true[support].ravel()) > 1e-12
    sign = float(np.mean(
        np.sign(A[support].ravel()[nz]) == np.sign(A_true[support].ravel()[nz])
    ))
    return {
        "support_average_precision": ap,
        "precision_at_k": precision,
        "recall_at_k": recall,
        "f1_at_k": f1,
        "truth_cosine": cosine_safe(score, truth_score),
        "truth_spearman": spearman_safe(score, truth_score),
        "sign_agreement_relevant": sign,
    }

def run_synthetic_ground_truth():
    out = ADDITIONAL_DATADIR / "additional_validation_synthetic_ground_truth_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    methods = ["ZeroOperator", "AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]
    C = 3
    for rep in range(SYN_REPEATS):
        for rho in SYN_RHOS:
            for outlier_rate in SYN_OUTLIER_LEVELS:
                seed = ADDITIONAL_SEED + rep * 10000 + int(rho * 1000) + int(outlier_rate * 100)
                X_clean, X_obs, Y, A_true, support, outlier_mask = make_inverse_ground_truth(
                    SYN_N, SYN_D, C, SYN_K, rho, outlier_rate, seed
                )
                idx = np.random.default_rng(seed + 2).permutation(SYN_N)
                cut = int(0.70 * SYN_N)
                tr, te = idx[:cut], idx[cut:]
                for method in methods:
                    if method == "ZeroOperator":
                        A = np.zeros_like(A_true)
                        state = {
                            "B": A.T, "weights": np.ones(len(tr)),
                            "residual_score": np.zeros(len(tr)),
                        }
                        diag = {
                            "cond_reg": np.nan, "coef_fro_norm": 0.0,
                            "frac_downweighted": 0.0, "mean_iter": 0,
                        }
                    else:
                        A, diag, state = fit_inverse_map(
                            X_obs[tr], Y[tr], method,
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
                            RESIDUAL_SCALE_MODE,
                        )
                    rec = support_metrics(A, A_true, support, SYN_K)
                    fidelity = inverse_reconstruction_metrics(
                        state, Y[te], X_clean[te], "clean_target_"
                    )
                    out_diag = outlier_weight_metrics(state, outlier_mask[tr])
                    rows.append({
                        "repeat": rep, "rho": rho, "outlier_rate": outlier_rate,
                        "method": method, "ridge_lambda": DEFAULT_RIDGE_LAMBDA,
                        "huber_delta": DEFAULT_HUBER_DELTA,
                        **diag, **rec, **fidelity, **out_diag,
                    })
    df = pd.DataFrame(rows)
    df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
    df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
    save_csv(df, out)
    return df

def synthetic_hra_ridge_summary(df):
    metrics = [
        "support_average_precision", "f1_at_k", "truth_cosine", "truth_spearman",
        "sign_agreement_relevant", "clean_target_reconstruction_r2",
        "clean_target_reconstruction_cosine", "log10_coef_norm",
    ]
    rows = []
    key = ["repeat", "rho", "outlier_rate"]
    for outlier_rate in sorted(df["outlier_rate"].unique()):
        sub = df[np.isclose(df["outlier_rate"], outlier_rate)]
        for metric in metrics:
            piv = sub.pivot_table(index=key, columns="method", values=metric, aggfunc="mean")
            if not {"RidgeAIME", "HuberRidgeAIME"}.issubset(piv.columns):
                continue
            piv = piv[["RidgeAIME", "HuberRidgeAIME"]].dropna().reset_index()
            piv["difference"] = piv["HuberRidgeAIME"] - piv["RidgeAIME"]
            # rho-by-repeat combinations are independently generated cells.
            cell_diff = piv.groupby(["repeat", "rho"])["difference"].mean()
            ci = cluster_bootstrap_ci(cell_diff, ADDITIONAL_CI_RESAMPLES, seed=ADDITIONAL_SEED + len(rows))
            _, p = wilcoxon_two_sided(cell_diff)
            rows.append({
                "outlier_rate": outlier_rate, "metric": metric,
                "n_generated_cells": ci["n_clusters"],
                "HRA_mean": float(piv["HuberRidgeAIME"].mean()),
                "RidgeAIME_mean": float(piv["RidgeAIME"].mean()),
                "HRA_minus_Ridge": ci["mean"],
                "CI95_low": ci["ci_low"], "CI95_high": ci["ci_high"],
                "wilcoxon_p": p,
            })
    return pd.DataFrame(rows)

SYNTHETIC_DISPLAY_METHODS = [
    "AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"
]
_synthetic_rates = np.asarray(sorted(float(v) for v in SYN_OUTLIER_LEVELS), float)
_synthetic_gap = float(np.min(np.diff(_synthetic_rates))) if len(_synthetic_rates) > 1 else 0.05
_synthetic_dodge_step = 0.04 * _synthetic_gap
SYNTHETIC_PLOT_HORIZONTAL_OFFSETS = {
    method: (index - 1.5) * _synthetic_dodge_step
    for index, method in enumerate(SYNTHETIC_DISPLAY_METHODS)
}
SYNTHETIC_PLOT_STYLES = {
    "AIME": {"color": "#1f77b4", "marker": "o", "linestyle": "--"},
    "HuberAIME": {"color": "#ff7f0e", "marker": "s", "linestyle": ":"},
    "RidgeAIME": {"color": "#2ca02c", "marker": "^", "linestyle": "-."},
    "HuberRidgeAIME": {"color": "#d62728", "marker": "D", "linestyle": "-"},
    "ZeroOperator": {"color": "#9467bd", "marker": "P", "linestyle": "--"},
}


def plot_synthetic_on_axis(ax, df, metric, ylabel, title=None):
    summary = (
        df.groupby(["method", "outlier_rate"], as_index=False)[metric]
        .agg(["mean", "std"])
        .reset_index()
    )
    for method in SYNTHETIC_DISPLAY_METHODS:
        g = summary[summary["method"] == method].sort_values("outlier_rate")
        if g.empty:
            continue
        nominal_x = g["outlier_rate"].to_numpy(float)
        displayed_x = nominal_x + SYNTHETIC_PLOT_HORIZONTAL_OFFSETS[method]
        style = SYNTHETIC_PLOT_STYLES[method]
        ax.errorbar(
            displayed_x,
            g["mean"].to_numpy(float),
            yerr=g["std"].to_numpy(float),
            marker=style["marker"],
            linestyle=style["linestyle"],
            color=style["color"],
            linewidth=1.4,
            markersize=4.5,
            capsize=3,
            label=method,
        )
    ax.set_xticks(_synthetic_rates)
    ax.set_xticklabels([f"{v:.2f}" for v in _synthetic_rates])
    x_padding = 0.16 * _synthetic_gap
    ax.set_xlim(float(_synthetic_rates.min() - x_padding), float(_synthetic_rates.max() + x_padding))
    ax.set_xlabel("Injected outlier rate π")
    ax.set_ylabel(ylabel)
    ax.set_title(title or "Known-ground-truth inverse-map recovery")
    ax.grid(axis="y", alpha=0.20, linewidth=0.6)
    ax.legend(fontsize=7, frameon=False)
    return summary


def plot_synthetic(df, metric, filename, ylabel):
    fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=220)
    plot_synthetic_on_axis(ax, df, metric, ylabel)
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)


def plot_synthetic_manuscript_figure(df):
    specs = [
        (
            "support_average_precision",
            "Support-recovery average precision",
            "Support recovery",
        ),
        (
            "clean_target_reconstruction_r2",
            "Clean held-out reconstruction R²",
            "Clean held-out reconstruction",
        ),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(15.0, 4.8), dpi=220)
    for panel_label, ax, (metric, ylabel, title) in zip("ab", axes, specs):
        plot_synthetic_on_axis(ax, df, metric, ylabel, title)
        ax.text(
            -0.10,
            1.05,
            panel_label,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="top",
        )
    fig.suptitle("Known-ground-truth inverse-map recovery", fontsize=14, y=1.02)
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S12_known_ground_truth.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path


ADDITIONAL_SYNTH_RAW = run_synthetic_ground_truth()
ADDITIONAL_SYNTH_CI = synthetic_hra_ridge_summary(ADDITIONAL_SYNTH_RAW)
save_table_bundle(
    ADDITIONAL_SYNTH_CI,
    "table_additional_validation_synthetic_ground_truth_ci",
    "Known-ground-truth inverse-map recovery. Coefficient norm is a diagnostic only; support and clean-target recovery are the faithfulness outcomes.",
    "tab:additional_validation_synthetic_ground_truth_ci",
)
plot_synthetic(
    ADDITIONAL_SYNTH_RAW, "support_average_precision",
    "fig_additional_validation_ground_truth_average_precision.png",
    "Support-recovery average precision",
)
plot_synthetic(
    ADDITIONAL_SYNTH_RAW, "clean_target_reconstruction_r2",
    "fig_additional_validation_ground_truth_clean_r2.png",
    "Clean held-out reconstruction R²",
)
ADDITIONAL_GROUND_TRUTH_MANUSCRIPT_FIGURE = plot_synthetic_manuscript_figure(ADDITIONAL_SYNTH_RAW)

def synthetic_method_quality_summary(df):
    metrics = [
        "coef_fro_norm", "support_average_precision", "f1_at_k", "truth_cosine",
        "sign_agreement_relevant", "clean_target_reconstruction_r2",
        "clean_target_reconstruction_cosine",
    ]
    return df.groupby(["method", "outlier_rate"])[metrics].agg(["mean", "std", "count"]).reset_index()

ADDITIONAL_SYNTH_METHOD_SUMMARY = synthetic_method_quality_summary(ADDITIONAL_SYNTH_RAW)
save_table_bundle(
    ADDITIONAL_SYNTH_METHOD_SUMMARY,
    "table_additional_validation_norm_is_not_quality",
    "Known-ground-truth control showing that coefficient norm is not a quality score. The zero operator has the smallest possible norm but poor support and reconstruction outcomes.",
    "tab:additional_validation_norm_is_not_quality",
)

ZERO_CONTROL_X_DISPLAY_FACTORS = {
    "RidgeAIME": 0.96,
    "HuberRidgeAIME": 1.04,
    "ZeroOperator": 1.00,
}
ZERO_CONTROL_STYLES = {
    "RidgeAIME": {"color": "#2ca02c", "marker": "^"},
    "HuberRidgeAIME": {"color": "#d62728", "marker": "D"},
    "ZeroOperator": {"color": "#9467bd", "marker": "P"},
}
fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=220)
plot_df = ADDITIONAL_SYNTH_RAW[
    ADDITIONAL_SYNTH_RAW["method"].isin(
        ["ZeroOperator", "RidgeAIME", "HuberRidgeAIME"]
    )
].copy()
for method in ["RidgeAIME", "HuberRidgeAIME", "ZeroOperator"]:
    g = plot_df[plot_df["method"] == method]
    displayed_x = g["coef_fro_norm"] * ZERO_CONTROL_X_DISPLAY_FACTORS[method]
    style = ZERO_CONTROL_STYLES[method]
    ax.scatter(
        displayed_x,
        g["support_average_precision"],
        alpha=0.45,
        s=28,
        marker=style["marker"],
        color=style["color"],
        label=method,
    )
ax.set_xscale("symlog", linthresh=1e-6)
ax.set_xlabel("Coefficient Frobenius norm (diagnostic only)")
ax.set_ylabel("Ground-truth support average precision")
ax.set_title("A smaller operator norm is not necessarily more faithful")
ax.text(
    0.99,
    0.02,
    "Nonzero x positions display-scaled by ±4% for visibility",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=7,
    color="0.35",
)
ax.legend(frameon=False)
fig.tight_layout()
path = ADDITIONAL_FIGDIR / "fig_additional_validation_norm_vs_ground_truth_faithfulness.png"
fig.savefig(path, bbox_inches="tight")
manuscript_path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S13_zero_operator_control.png"
fig.savefig(manuscript_path, bbox_inches="tight")
plt.close(fig)
print("[figure]", path)
print("[figure]", manuscript_path)
ADDITIONAL_ZERO_OPERATOR_MANUSCRIPT_FIGURE = manuscript_path

display(ADDITIONAL_SYNTH_CI)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/additional_validation_synthetic_ground_truth_raw.csv
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_synthetic_ground_truth_ci.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_synthetic_ground_truth_ci.tex
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/figures/fig_additional_validation_ground_truth_average_precision.png
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/figures/fig_additional_validation_ground_truth_clean_r2.png
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebook

,outlier_rate,metric,n_generated_cells,HRA_mean,RidgeAIME_mean,HRA_minus_Ridge,CI95_low,CI95_high,wilcoxon_p
0,0.00,support_average_precision,60,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000e+00
1,0.00,f1_at_k,60,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000e+00
2,0.00,truth_cosine,60,0.999185,0.999185,0.000000,0.000000,0.000000,1.000000e+00
3,0.00,truth_spearman,60,0.520593,0.520593,0.000000,0.000000,0.000000,1.000000e+00
4,0.00,sign_agreement_relevant,60,0.997778,0.997778,0.000000,0.000000,0.000000,1.000000e+00
5,0.00,clean_target_reconstruction_r2,60,0.569597,0.569597,0.000000,0.000000,0.000000,1.000000e+00
6,0.00,clean_target_reconstruction_cosine,60,0.959379,0.959379,0.000000,0.000000,0.000000,1.000000e+00
7,0.00,log10_coef_norm,60,0.685642,0.685642,0.000000,0.000000,0.000000,1.000000e+00
8,0.05,support_average_precision,60,1.000000,0.871882,0.128118,0.081400,0.178963,5.597410e-06
9,0.05,f1_at_k,60,1.000000,0.835000,0.165000,0.111625,0.221667,5.010872e-06


## Experiment 3 — \(\lambda\) and \(\delta\) sensitivity, including true decoys

The submitted setting is reported explicitly as \(\lambda=0.01\), \(\delta=1.0\), with the threshold applied to row RMS input-space residuals. Under joint high stress, the notebook tracks:

- conditioning,
- coefficient norm (**diagnostic only**),
- clean-target inverse reconstruction,
- recovery of the clean-reference operator,
- mass assigned to correlated clone features,
- mass assigned to independently permuted **irrelevant decoys**,
- Huber detection of known injected-outlier rows,
- bootstrap stability.

This directly tests the sensitivity-analysis hypothesis that a smaller \(\lambda\) may retain much of the conditioning benefit while reducing the decoy cost.


In [12]:
# %% [Experiment 3: lambda/delta sensitivity with true decoys]
def build_hyperparameter_report():
    rows = []
    for dataset in DATASETS:
        for learner in LEARNERS:
            rows.append({
                "dataset": dataset, "learner": learner,
                "submitted_ridge_lambda": DEFAULT_RIDGE_LAMBDA,
                "submitted_huber_delta": DEFAULT_HUBER_DELTA,
                "residual_definition": "row RMS residual = L2 residual / sqrt(d)",
                "selection_rule": "fixed globally before this sensitivity analysis",
            })
    df = pd.DataFrame(rows)
    save_table_bundle(df, "table_additional_validation_hyperparameters_used",
        "Hyperparameter values and the residual definition used in the additional validation experiments.",
        "tab:additional_validation_hyperparameters_used")
    return df

def run_sensitivity():
    out = ADDITIONAL_DATADIR / "additional_validation_lambda_delta_sensitivity_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    for dataset in DATASETS:
        X, y, _ = standardized_dataset(dataset, ADDITIONAL_SEED)
        for rep in range(SENSITIVITY_REPEATS):
            for learner in LEARNERS:
                seed_model = ADDITIONAL_SEED + rep * 10000 + 100 * (LEARNERS.index(learner) + 1)
                print(f"[sensitivity] {dataset} {learner} rep={rep}")
                model, _, exp_idx, Y_exp, model_info = fit_clean_blackbox(X, y, learner, seed_model)
                X_exp, y_exp = X[exp_idx], y[exp_idx]
                X_clean_aug, X_stress_aug, out_mask, _, clone_idx, info = build_stress_pair(
                    X_exp, 0.10, 0.50, seed_model + 777
                )
                fit_idx, val_idx = split_explanation_rows(y_exp, seed_model + 778)
                for lam in LAMBDA_GRID:
                    A_ref, _, _ = fit_inverse_map(
                        X_clean_aug[fit_idx], Y_exp[fit_idx], "RidgeAIME",
                        DEFAULT_HUBER_DELTA, lam, RESIDUAL_SCALE_MODE,
                    )
                    for method, deltas in [("RidgeAIME", [np.nan]), ("HuberRidgeAIME", DELTA_GRID)]:
                        for delta_value in deltas:
                            delta = DEFAULT_HUBER_DELTA if np.isnan(delta_value) else float(delta_value)
                            A, diag, state = fit_inverse_map(
                                X_stress_aug[fit_idx], Y_exp[fit_idx], method, delta, lam, RESIDUAL_SCALE_MODE,
                            )
                            clean_rec = inverse_reconstruction_metrics(state, Y_exp[val_idx], X_clean_aug[val_idx], "clean_target_")
                            op = operator_recovery_metrics(A, A_ref)
                            clone = subset_mass_metrics(A, clone_idx, "clone")
                            decoy = irrelevant_decoy_metrics(
                                X_stress_aug[fit_idx], Y_exp[fit_idx], method, delta, lam,
                                SENS_DECOY_TRIALS, seed_model + 880,
                            )
                            out_diag = outlier_weight_metrics(state, out_mask[fit_idx])
                            boot = bootstrap_inverse_stability(
                                X_stress_aug[fit_idx], Y_exp[fit_idx], method, delta, lam,
                                A0=A, B=SENS_BOOTSTRAPS, seed=seed_model + 779,
                            )
                            rows.append({
                                "dataset": dataset, "learner": learner, "repeat": rep,
                                "method": method, "ridge_lambda": lam,
                                "huber_delta": np.nan if method == "RidgeAIME" else delta,
                                **info, **model_info, **diag, **clean_rec, **op, **clone, **decoy,
                                **out_diag, **boot,
                            })
    df = pd.DataFrame(rows)
    df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
    df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
    save_csv(df, out)
    return df

def summarize_sensitivity(df):
    metrics = [
        "log10_cond_reg", "log10_coef_norm",
        "clean_target_reconstruction_r2", "clean_target_reconstruction_cosine",
        "operator_cosine_flat", "clone_mass_ratio", "clone_topk_infiltration",
        "irrelevant_decoy_mass_ratio", "irrelevant_decoy_topk_infiltration",
        "outlier_weight_ap", "bootstrap_cosine_flat", "frac_downweighted",
    ]
    return df.groupby(["method", "ridge_lambda", "huber_delta"], dropna=False)[metrics].agg(["mean", "std", "count"]).reset_index()

def plot_lambda_on_axis(ax, df, metric, ylabel, method="HuberRidgeAIME", title=None):
    sub = df[df["method"] == method].copy()
    if method == "HuberRidgeAIME":
        for delta, g in sub.groupby("huber_delta"):
            summary = g.groupby("ridge_lambda")[metric].mean().reset_index()
            ax.plot(
                summary["ridge_lambda"].replace(0, 1e-7),
                summary[metric],
                marker="o",
                markersize=3.5,
                linewidth=1.2,
                label=f"delta={delta:g}",
            )
    else:
        summary = sub.groupby("ridge_lambda")[metric].mean().reset_index()
        ax.plot(
            summary["ridge_lambda"].replace(0, 1e-7),
            summary[metric],
            marker="o",
            markersize=3.5,
            linewidth=1.2,
            label=method,
        )
    ax.axvline(
        DEFAULT_RIDGE_LAMBDA,
        linestyle="--",
        linewidth=1,
        color="0.35",
        label="default lambda",
    )
    ax.set_xscale("log")
    ax.set_xlabel("Ridge parameter λ (0 plotted at 10⁻⁷)")
    ax.set_ylabel(ylabel)
    ax.set_title(title or f"{method}: {metric}")
    ax.grid(axis="y", alpha=0.20, linewidth=0.6)
    ax.legend(fontsize=7, frameon=False)


def plot_lambda(df, metric, filename, ylabel, method="HuberRidgeAIME"):
    fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=220)
    plot_lambda_on_axis(ax, df, metric, ylabel, method, ylabel)
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)


def plot_sensitivity_manuscript_figure(df):
    specs = [
        ("log10_cond_reg", "log₁₀ condition number", "Conditioning"),
        (
            "clean_target_reconstruction_cosine",
            "Clean-target reconstruction cosine",
            "Clean-target reconstruction",
        ),
        (
            "operator_cosine_flat",
            "Clean-reference operator cosine",
            "Clean-reference operator recovery",
        ),
        (
            "irrelevant_decoy_mass_ratio",
            "Irrelevant-decoy attribution mass",
            "Irrelevant-decoy attribution",
        ),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(15.0, 9.2), dpi=220)
    for panel_label, ax, (metric, ylabel, title) in zip("abcd", axes.ravel(), specs):
        plot_lambda_on_axis(ax, df, metric, ylabel, title=title)
        ax.text(
            -0.10,
            1.05,
            panel_label,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="top",
        )
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S14_lambda_delta_sensitivity.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path


ADDITIONAL_HYPERPARAMETERS = build_hyperparameter_report()
ADDITIONAL_SENS_RAW = run_sensitivity()
ADDITIONAL_SENS_SUMMARY = summarize_sensitivity(ADDITIONAL_SENS_RAW)
save_table_bundle(ADDITIONAL_SENS_SUMMARY, "table_additional_validation_lambda_delta_sensitivity",
    "Sensitivity of conditioning, clean-target reconstruction, clean-reference operator recovery, irrelevant-decoy mass, clone mass, stability, and Huber row weighting to lambda and delta under joint high stress.",
    "tab:additional_validation_lambda_delta_sensitivity")
plot_lambda(ADDITIONAL_SENS_RAW, "log10_cond_reg", "fig_additional_validation_lambda_conditioning.png", "log10 condition number")
plot_lambda(ADDITIONAL_SENS_RAW, "irrelevant_decoy_mass_ratio", "fig_additional_validation_lambda_irrelevant_decoy_mass.png", "Irrelevant-decoy attribution mass")
plot_lambda(ADDITIONAL_SENS_RAW, "clone_mass_ratio", "fig_additional_validation_lambda_clone_mass.png", "Correlated-clone attribution mass")
plot_lambda(ADDITIONAL_SENS_RAW, "clean_target_reconstruction_cosine", "fig_additional_validation_lambda_clean_reconstruction.png", "Clean-target reconstruction cosine")
plot_lambda(ADDITIONAL_SENS_RAW, "operator_cosine_flat", "fig_additional_validation_lambda_operator_recovery.png", "Clean-reference operator cosine")
ADDITIONAL_SENSITIVITY_MANUSCRIPT_FIGURE = plot_sensitivity_manuscript_figure(ADDITIONAL_SENS_RAW)
display(ADDITIONAL_SENS_SUMMARY)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_hyperparameters_used.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_hyperparameters_used.tex
[sensitivity] breast_cancer lgbm rep=0
[sensitivity] breast_cancer svm rep=0
[sensitivity] breast_cancer mlp rep=0
[sensitivity] breast_cancer lgbm rep=1
[sensitivity] breast_cancer svm rep=1
[sensitivity] breast_cancer mlp rep=1
[sensitivity] breast_cancer lgbm rep=2
[sensitivity] breast_cancer svm rep=2
[sensitivity] breast_cancer mlp rep=2
[sensitivity] credit_approval lgbm rep=0
[sensitivity] credit_approval svm rep=0
[sensitivity] credit_approval mlp rep=0
[sensitivity] credit_approval lgbm rep=1
[sensitivity] credit_approval svm rep=1
[sensitivity] credit_approval mlp rep=1
[sensitivity] credit_approval lgbm rep=2
[sensitivity

method ridge_lambda huber_delta log10_cond_reg                  \
                                                      mean       std count   
0   HuberRidgeAIME     0.000000         0.5       0.343600  0.134381    27   
1   HuberRidgeAIME     0.000000         1.0       0.275490  0.151827    27   
2   HuberRidgeAIME     0.000000         2.0       0.266445  0.156033    27   
3   HuberRidgeAIME     0.000001         0.5       0.343600  0.134381    27   
4   HuberRidgeAIME     0.000001         1.0       0.275490  0.151827    27   
5   HuberRidgeAIME     0.000001         2.0       0.266445  0.156033    27   
6   HuberRidgeAIME     0.000010         0.5       0.343600  0.134380    27   
7   HuberRidgeAIME     0.000010         1.0       0.275490  0.151827    27   
8   HuberRidgeAIME     0.000010         2.0       0.266445  0.156033    27   
9   HuberRidgeAIME     0.000100         0.5       0.343599  0.134380    27   
10  HuberRidgeAIME     0.000100         1.0       0.275490  0.151826    27   
11  HuberRidgeAIME     0.000100         2.0       0.266444  0.156032    27   
12  HuberRidgeAIME     0.001000         0.5       0.343589  0.134372    27   
13  HuberRidgeAIME     0.001000         1.0       0.275484  0.151820    27   
14  HuberRidgeAIME     0.001000         2.0       0.266439  0.156026    27   
15  HuberRidgeAIME     0.010000         0.5       0.343490  0.134299    27   
16  HuberRidgeAIME     0.010000         1.0       0.275427  0.151759    27   
17  HuberRidgeAIME     0.010000         2.0       0.266384  0.155967    27   
18  HuberRidgeAIME     0.100000         0.5       0.342500  0.133574    27   
19  HuberRidgeAIME     0.100000         1.0       0.274858  0.151149    27   
20  HuberRidgeAIME     0.100000         2.0       0.265840  0.155377    27   
21  HuberRidgeAIME     1.000000         0.5       0.333132  0.126879    27   
22  HuberRidgeAIME     1.000000         1.0       0.269335  0.145320    27   
23  HuberRidgeAIME     1.000000         2.0       0.260552  0.149732    27   
24       RidgeAIME     0.000000         1.0       0.267494  0.157154    27   
25       RidgeAIME     0.000001         1.0       0.267494  0.157154    27   
26       RidgeAIME     0.000010         1.0       0.267494  0.157154    27   
27       RidgeAIME     0.000100         1.0       0.267493  0.157154    27   
28       RidgeAIME     0.001000         1.0       0.267488  0.157148    27   
29       RidgeAIME     0.010000         1.0       0.267435  0.157091    27   
30       RidgeAIME     0.100000         1.0       0.266912  0.156522    27   
31       RidgeAIME     1.000000         1.0       0.261821  0.151075    27   

   log10_coef_norm                 clean_target_reconstruction_r2  ...  \
              mean       std count                           mean  ...   
0         1.023358  0.468013    27                       0.283095  ...   
1         1.028919  0.469351    27                       0.284986  ...   
2         1.041559  0.465882    27                       0.281446  ...   
3         1.023358  0.468013    27                       0.283095  ...   
4         1.028919  0.469351    27                       0.284986  ...   
5         1.041559  0.465882    27                       0.281446  ...   
6         1.023358  0.468013    27                       0.283095  ...   
7         1.028919  0.469351    27                       0.284986  ...   
8         1.041559  0.465882    27                       0.281446  ...   
9         1.023356  0.468014    27                       0.283095  ...   
10        1.028918  0.469351    27                       0.284986  ...   
11        1.041558  0.465882    27                       0.281446  ...   
12        1.023341  0.468019    27                       0.283095  ...   
13        1.028907  0.469354    27                       0.284986  ...   
14        1.041548  0.465885    27                       0.281447  ...   
15        1.023182  0.468067    27                       0.283092  ...   
16        1.028800  0.46938

## Experiment 4 — Main-text LIME and SHAP comparison under stress

A representative LightGBM comparison is run because it permits genuine native TreeSHAP without the external `shap`/Numba dependency that failed in the first run.

- Full execution requires the standard `lime.lime_tabular.LimeTabularExplainer` package; the internal surrogate is allowed only in quick-test mode.
- SHAP uses LightGBM native TreeSHAP (`pred_contrib=True`).
- Failed rows are never summarized: any error aborts the baseline table.
- Both condition-wise summaries and paired clean-to-stress changes are generated.

Correlated-clone mass is descriptive because the stressed black box is trained with those features; it is not called irrelevant-decoy faithfulness.


In [13]:

# %% [Experiment 4: internal LIME and native TreeSHAP]
def fit_model_on_condition(X, y, learner, seed):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)), test_size=0.30, random_state=seed, stratify=y
    )
    model = make_learner(learner, seed)
    t0 = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_time = time.perf_counter() - t0
    Y = model.predict_proba(X[test_idx])
    pred_idx = np.argmax(Y, axis=1)
    classes = model_classes(model)
    pred = classes[pred_idx] if classes is not None else pred_idx
    return model, train_idx, test_idx, Y, {
        "model_fit_time_sec": fit_time,
        "model_test_accuracy": float(accuracy_score(y[test_idx], pred)),
    }

def weighted_ridge_with_intercept(Z, target, weights, ridge=1e-3):
    Z = np.asarray(Z, float)
    target = np.asarray(target, float)
    weights = np.asarray(weights, float)
    design = np.column_stack([np.ones(len(Z)), Z])
    G = design.T @ (design * weights[:, None])
    G[1:, 1:] += ridge * np.eye(Z.shape[1])
    rhs = design.T @ (target * weights)
    try:
        beta = np.linalg.solve(G, rhs)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(G) @ rhs
    return beta[0], beta[1:]

def _lime_local_matrix_internal(model, X_train, X_eval, seed, n_samples=LIME_NUM_SAMPLES):
    rng = np.random.default_rng(seed)
    X_train = np.asarray(X_train, float)
    X_eval = np.asarray(X_eval, float)
    center = X_train.mean(axis=0)
    scale = np.where(X_train.std(axis=0) < 1e-12, 1.0, X_train.std(axis=0))
    lo, hi = np.quantile(X_train, 0.01, axis=0), np.quantile(X_train, 0.99, axis=0)
    d = X_train.shape[1]
    width = 0.75 * np.sqrt(d)
    local = []
    for x in X_eval:
        Z = x + rng.normal(size=(n_samples, d)) * scale
        Z[0] = x
        Z = np.clip(Z, lo, hi)
        probs = model.predict_proba(Z)
        cls = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
        Zs = (Z - x) / scale
        dist2 = np.sum(Zs ** 2, axis=1)
        weights = np.exp(-dist2 / max(width ** 2, 1e-12))
        _, coef = weighted_ridge_with_intercept(Zs, probs[:, cls], weights, LIME_RIDGE)
        contribution = coef * ((x - center) / scale)
        local.append(contribution)
    return np.asarray(local)


def lime_local_matrix(model, X_train, X_eval, seed, n_samples=LIME_NUM_SAMPLES):
    """Return one local coefficient vector per evaluated row."""
    if not LIME_PACKAGE_AVAILABLE:
        return _lime_local_matrix_internal(model, X_train, X_eval, seed, n_samples)
    X_train = np.asarray(X_train, float)
    X_eval = np.asarray(X_eval, float)
    d = X_train.shape[1]
    explainer = LimeTabularExplainer(
        X_train,
        mode="classification",
        discretize_continuous=False,
        random_state=seed,
        feature_names=[f"x{i}" for i in range(d)],
    )
    local = []
    for x in X_eval:
        cls = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
        exp = explainer.explain_instance(
            x, model.predict_proba, labels=[cls],
            num_features=min(d, LIME_NUM_FEATURES), num_samples=n_samples,
        )
        vec = np.zeros(d)
        for j, weight in exp.as_map().get(cls, []):
            vec[int(j)] = float(weight)
        local.append(vec)
    return np.asarray(local)

def tree_shap_local_matrix(model, X_eval):
    if not LGBM_AVAILABLE or "lgbm" not in model.__class__.__name__.lower():
        raise RuntimeError("Native TreeSHAP requires a genuine LightGBM model.")
    raw = np.asarray(model.predict(X_eval, pred_contrib=True))
    d = X_eval.shape[1]
    probs = model.predict_proba(X_eval)
    if raw.shape[1] == d + 1:  # binary
        return raw[:, :d]
    C = probs.shape[1]
    if raw.shape[1] != C * (d + 1):
        raise ValueError(f"Unexpected LightGBM contribution shape {raw.shape}")
    arr = raw.reshape(len(X_eval), C, d + 1)
    cls = np.argmax(probs, axis=1)
    return np.stack([arr[i, cls[i], :d] for i in range(len(X_eval))], axis=0)

def local_matrix_stability(local, B, seed):
    rng = np.random.default_rng(seed)
    local = np.asarray(local, float)
    g0 = np.mean(np.abs(local), axis=0)
    vals = []
    for _ in range(B):
        idx = rng.choice(len(local), size=len(local), replace=True)
        gb = np.mean(np.abs(local[idx]), axis=0)
        vals.append({
            "bootstrap_cosine": cosine_safe(g0, gb),
            "bootstrap_spearman": spearman_safe(g0, gb),
            "bootstrap_topk": topk_jaccard(g0, gb, TOP_K),
        })
    return pd.DataFrame(vals).mean().to_dict()

def noise_local_stability(method, model, X_train, X_eval, local0, seed):
    rng = np.random.default_rng(seed)
    col_sd = np.where(X_train.std(axis=0) < 1e-12, 1.0, X_train.std(axis=0))
    g0 = np.mean(np.abs(local0), axis=0)
    vals = []
    for r in range(BASELINE_NOISE_REPEATS):
        Xn = X_eval + rng.normal(0, 0.02, size=X_eval.shape) * col_sd
        if method == "LIME":
            localn = lime_local_matrix(model, X_train, Xn, seed + 100 + r)
        else:
            localn = tree_shap_local_matrix(model, Xn)
        gn = np.mean(np.abs(localn), axis=0)
        vals.append({
            "noise_cosine": cosine_safe(g0, gn),
            "noise_spearman": spearman_safe(g0, gn),
            "noise_topk": topk_jaccard(g0, gn, TOP_K),
        })
    return pd.DataFrame(vals).mean().to_dict()

def aime_baseline_metrics(X_test, Y_test, model, method, seed):
    t0 = time.perf_counter()
    A, diag, state = fit_inverse_map(
        X_test, Y_test, method,
        DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
    )
    runtime = time.perf_counter() - t0
    boot = bootstrap_inverse_stability(
        X_test, Y_test, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
        A0=A, B=BASELINE_BOOTSTRAPS, seed=seed,
    )
    rng = np.random.default_rng(seed + 1)
    col_sd = np.where(X_test.std(axis=0) < 1e-12, 1.0, X_test.std(axis=0))
    noise_vals = []
    for _ in range(BASELINE_NOISE_REPEATS):
        Xn = X_test + rng.normal(0, 0.02, size=X_test.shape) * col_sd
        Yn = model.predict_proba(Xn)
        An, _, _ = fit_inverse_map(
            Xn, Yn, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
            RESIDUAL_SCALE_MODE,
        )
        noise_vals.append({
            "noise_cosine": cosine_safe(A, An),
            "noise_spearman": spearman_safe(global_strength(A), global_strength(An)),
            "noise_topk": topk_jaccard(global_strength(A), global_strength(An), TOP_K),
        })
    noise = pd.DataFrame(noise_vals).mean().to_dict()
    return A, runtime, boot, noise, diag

def run_baseline_comparison():
    out = ADDITIONAL_DATADIR / "additional_validation_lime_shap_stress_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_BASELINES:
        return pd.DataFrame()

    rows = []
    for dataset in BASELINE_DATASETS:
        X_clean, y, meta = standardized_dataset(dataset, ADDITIONAL_SEED)
        d0 = X_clean.shape[1]
        for rep in range(BASELINE_REPEATS):
            for outlier_rate, clone_frac in BASELINE_CONDITIONS:
                seed = ADDITIONAL_SEED + rep * 10000 + int(outlier_rate * 1000) + int(clone_frac * 100)
                # Here the stressed black box is trained on the same stressed feature matrix used by all explainers.
                _, X_condition, _, _, clone_idx, info = build_stress_pair(
                    X_clean, outlier_rate, clone_frac, seed
                )
                model, train_idx, test_idx, Y_test, model_info = fit_model_on_condition(
                    X_condition, y, BASELINE_LEARNER, seed
                )
                X_train, X_test = X_condition[train_idx], X_condition[test_idx]
                rng = np.random.default_rng(seed + 7)
                eval_idx = rng.choice(
                    len(X_test), size=min(BASELINE_EVAL_N, len(X_test)), replace=False
                )
                X_eval = X_test[eval_idx]

                for method in ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]:
                    A, runtime, boot, noise, diag = aime_baseline_metrics(
                        X_test, Y_test, model, method, seed + 20
                    )
                    g = global_strength(A)
                    clone_mass = float(g[clone_idx].sum() / (g.sum() + 1e-12)) if clone_idx else 0.0
                    rows.append({
                        "dataset": dataset, "learner": BASELINE_LEARNER, "repeat": rep,
                        "condition": "clean" if outlier_rate == 0 else "joint_high_stress",
                        "method": method, **info, **model_info,
                        "explain_time_sec": runtime,
                        "added_clone_mass": clone_mass,
                        "bootstrap_cosine": boot["bootstrap_cosine_flat"],
                        "bootstrap_spearman": boot["bootstrap_spearman_global"],
                        "bootstrap_topk": boot["bootstrap_topk_jaccard"],
                        **noise, "implementation": "equation-consistent inverse map",
                        "error": "",
                    })

                for method in ["LIME", "SHAP_TreeSHAP"]:
                    t0 = time.perf_counter()
                    try:
                        if method == "LIME":
                            local = lime_local_matrix(model, X_train, X_eval, seed + 30)
                        else:
                            local = tree_shap_local_matrix(model, X_eval)
                        runtime = time.perf_counter() - t0
                        g = np.mean(np.abs(local), axis=0)
                        boot = local_matrix_stability(local, BASELINE_BOOTSTRAPS, seed + 31)
                        noise = noise_local_stability(
                            "LIME" if method == "LIME" else "SHAP",
                            model, X_train, X_eval, local, seed + 32,
                        )
                        clone_mass = float(g[clone_idx].sum() / (g.sum() + 1e-12)) if clone_idx else 0.0
                        implementation = (
                            LIME_IMPLEMENTATION
                            if method == "LIME"
                            else "LightGBM native TreeSHAP pred_contrib=True"
                        )
                        rows.append({
                            "dataset": dataset, "learner": BASELINE_LEARNER, "repeat": rep,
                            "condition": "clean" if outlier_rate == 0 else "joint_high_stress",
                            "method": method, **info, **model_info,
                            "explain_time_sec": runtime,
                            "added_clone_mass": clone_mass,
                            **boot, **noise,
                            "implementation": implementation,
                            "error": "",
                        })
                    except Exception as e:
                        rows.append({
                            "dataset": dataset, "learner": BASELINE_LEARNER, "repeat": rep,
                            "condition": "clean" if outlier_rate == 0 else "joint_high_stress",
                            "method": method, **info, **model_info,
                            "error": repr(e),
                        })

    df = pd.DataFrame(rows)
    save_csv(df, out)
    return df

def summarize_baselines(df):
    expected_methods = {"AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME", "LIME", "SHAP_TreeSHAP"}
    errors = df[df["error"].fillna("") != ""][["dataset", "method", "condition", "error"]]
    if len(errors):
        save_csv(errors, ADDITIONAL_LOGDIR / "additional_validation_baseline_errors.csv")
        raise RuntimeError("At least one baseline row failed; failed rows are never summarized. See logs/additional_validation_baseline_errors.csv")
    valid = df.copy()
    missing = expected_methods - set(valid["method"].unique())
    if missing:
        raise AssertionError(f"Missing baseline methods: {sorted(missing)}")
    if (valid["method"] == "SHAP_TreeSHAP").sum() == 0:
        raise AssertionError("No valid native TreeSHAP rows were produced.")
    metrics = [
        "bootstrap_cosine", "bootstrap_spearman", "bootstrap_topk",
        "noise_cosine", "noise_spearman", "noise_topk",
        "added_clone_mass", "explain_time_sec",
    ]
    summary = valid.groupby(["method", "condition"])[metrics].agg(["mean", "std", "count"]).reset_index()
    return summary

def baseline_stress_change_summary(df):
    metrics = ["bootstrap_cosine", "noise_cosine", "added_clone_mass", "explain_time_sec"]
    rows=[]
    for metric in metrics:
        piv=df.pivot_table(index=["dataset","repeat","method"], columns="condition", values=metric, aggfunc="mean")
        if not {"clean","joint_high_stress"}.issubset(piv.columns):
            continue
        piv=piv.dropna().reset_index(); piv["stress_minus_clean"]=piv["joint_high_stress"]-piv["clean"]
        for method,g in piv.groupby("method"):
            # dataset-level clusters; repeats are averaged before bootstrap
            cell=g.groupby("dataset")["stress_minus_clean"].mean()
            ci=cluster_bootstrap_ci(cell, ADDITIONAL_CI_RESAMPLES, seed=ADDITIONAL_SEED+len(rows))
            rows.append({
                "metric":metric,"method":method,"n_dataset_clusters":ci["n_clusters"],
                "stress_minus_clean_mean":ci["mean"],"CI95_low":ci["ci_low"],"CI95_high":ci["ci_high"],
            })
    return pd.DataFrame(rows)

ADDITIONAL_BASELINE_RAW = run_baseline_comparison()
if len(ADDITIONAL_BASELINE_RAW):
    ADDITIONAL_BASELINE_SUMMARY = summarize_baselines(ADDITIONAL_BASELINE_RAW)
    ADDITIONAL_BASELINE_CHANGE = baseline_stress_change_summary(ADDITIONAL_BASELINE_RAW)
    save_table_bundle(ADDITIONAL_BASELINE_SUMMARY, "table_additional_validation_lime_shap_stress",
        "Representative equation-consistent AIME-family, standard LIME, and native TreeSHAP comparison under clean and joint-high-stress LightGBM conditions.",
        "tab:additional_validation_lime_shap_stress")
    save_table_bundle(ADDITIONAL_BASELINE_CHANGE, "table_additional_validation_lime_shap_stress_changes",
        "Paired clean-to-stress changes for the representative AIME-family, LIME, and native TreeSHAP comparison.",
        "tab:additional_validation_lime_shap_stress_changes")
    display(ADDITIONAL_BASELINE_SUMMARY); display(ADDITIONAL_BASELINE_CHANGE)
else:
    ADDITIONAL_BASELINE_SUMMARY = pd.DataFrame(); ADDITIONAL_BASELINE_CHANGE = pd.DataFrame()
    print("Baseline comparison skipped by configuration.")


BASELINE_DISPLAY_METHODS = [
    "AIME",
    "HuberAIME",
    "RidgeAIME",
    "HuberRidgeAIME",
    "LIME",
    "SHAP_TreeSHAP",
]
BASELINE_DISPLAY_LABELS = {
    "AIME": "AIME",
    "HuberAIME": "HuberAIME",
    "RidgeAIME": "RidgeAIME",
    "HuberRidgeAIME": "HRA",
    "LIME": "LIME",
    "SHAP_TreeSHAP": "TreeSHAP",
}
BASELINE_METHOD_STYLES = {
    "AIME": {"color": "#1f77b4", "marker": "o"},
    "HuberAIME": {"color": "#ff7f0e", "marker": "s"},
    "RidgeAIME": {"color": "#2ca02c", "marker": "^"},
    "HuberRidgeAIME": {"color": "#d62728", "marker": "D"},
    "LIME": {"color": "#9467bd", "marker": "P"},
    "SHAP_TreeSHAP": {"color": "#8c564b", "marker": "X"},
}


def plot_baseline_change_on_axis(ax, change_df, metric, ylabel, title):
    subset = change_df[change_df["metric"] == metric].set_index("method")
    positions = np.arange(len(BASELINE_DISPLAY_METHODS))
    for position, method in zip(positions, BASELINE_DISPLAY_METHODS):
        if method not in subset.index:
            continue
        row = subset.loc[method]
        value = float(row["stress_minus_clean_mean"])
        low = max(0.0, value - float(row["CI95_low"]))
        high = max(0.0, float(row["CI95_high"]) - value)
        style = BASELINE_METHOD_STYLES[method]
        ax.errorbar(
            [position],
            [value],
            yerr=np.asarray([[low], [high]]),
            color=style["color"],
            marker=style["marker"],
            markersize=6,
            capsize=4,
            linestyle="none",
        )
    ax.axhline(0, color="0.35", linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(
        [BASELINE_DISPLAY_LABELS[method] for method in BASELINE_DISPLAY_METHODS],
        rotation=25,
        ha="right",
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.20, linewidth=0.6)


def plot_baseline_manuscript_figure(change_df):
    specs = [
        (
            "bootstrap_cosine",
            "Stress minus clean bootstrap cosine",
            "Bootstrap stability change",
        ),
        (
            "noise_cosine",
            "Stress minus clean noise cosine",
            "Noise-agreement change",
        ),
        (
            "added_clone_mass",
            "Stress minus clean clone mass",
            "Added-clone attribution mass",
        ),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(17.0, 4.9), dpi=220)
    for panel_label, ax, (metric, ylabel, title) in zip("abc", axes, specs):
        plot_baseline_change_on_axis(ax, change_df, metric, ylabel, title)
        ax.text(
            -0.12,
            1.05,
            panel_label,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="top",
        )
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S15_lime_treeshap_stress.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path


if len(ADDITIONAL_BASELINE_CHANGE):
    ADDITIONAL_BASELINE_MANUSCRIPT_FIGURE = plot_baseline_manuscript_figure(
        ADDITIONAL_BASELINE_CHANGE
    )
else:
    ADDITIONAL_BASELINE_MANUSCRIPT_FIGURE = None


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/additional_validation_lime_shap_stress_raw.csv
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_lime_shap_stress.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_lime_shap_stress.tex
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_lime_shap_stress_changes.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_lime_shap_stress_changes.tex


method          condition bootstrap_cosine                  \
                                                  mean       std count   
0             AIME              clean         0.972120  0.031370     6   
1             AIME  joint_high_stress         0.932335  0.055874     6   
2        HuberAIME              clean         0.975070  0.027674     6   
3        HuberAIME  joint_high_stress         0.976415  0.023958     6   
4   HuberRidgeAIME              clean         0.975070  0.027675     6   
5   HuberRidgeAIME  joint_high_stress         0.976415  0.023958     6   
6             LIME              clean         0.946510  0.081440     6   
7             LIME  joint_high_stress         0.958246  0.060455     6   
8        RidgeAIME              clean         0.972120  0.031371     6   
9        RidgeAIME  joint_high_stress         0.932334  0.055875     6   
10   SHAP_TreeSHAP              clean         0.970353  0.038676     6   
11   SHAP_TreeSHAP  joint_high_stress         0.968206  0.041411     6   

   bootstrap_spearman                 bootstrap_topk            ...  \
                 mean       std count           mean       std  ...   
0            0.916196  0.096323     6       0.665783  0.133352  ...   
1            0.828821  0.145679     6       0.485323  0.164446  ...   
2            0.931198  0.082753     6       0.701676  0.130087  ...   
3            0.918290  0.085729     6       0.638496  0.190459  ...   
4            0.931180  0.082790     6       0.701676  0.130087  ...   
5            0.918270  0.085758     6       0.638496  0.190459  ...   
6            0.864757  0.075091     6       0.716664  0.279873  ...   
7            0.882421  0.054524     6       0.770252  0.186002  ...   
8            0.916196  0.096324     6       0.665783  0.133352  ...   
9            0.828821  0.145679     6       0.485323  0.164446  ...   
10           0.971637  0.020081     6       0.801465  0.160454  ...   
11           0.967185  0.031188     6       0.782617  0.155975  ...   

   noise_spearman noise_topk                 added_clone_mass                  \
            count       mean       std count             mean       std count   
0               6   0.898990  0.049485     6         0.000000  0.000000     6   
1               6   0.949495  0.045623     6         0.291468  0.116719     6   
2               6   0.939394  0.076661     6         0.000000  0.000000     6   
3               6   0.910774  0.096551     6         0.293488  0.120227     6   
4               6   0.939394  0.076661     6         0.000000  0.000000     6   
5               6   0.910774  0.096551     6         0.293488  0.120227     6   
6               6   0.672302  0.292539     6         0.000000  0.000000     6   
7               6   0.772857  0.169268     6         0.310355  0.178391     6   
8               6   0.898990  0.049485     6         0.000000  0.000000     6   
9               6   0.949495  0.045623     6         0.291468  0.116719     6   
10              6   0.944056  0.087118     6         0.000000  0.000000     6   
11              6   0.856514  0.113663     6         0.359679  0.267273     6   

   explain_time_sec                  
               mean       std count  
0          0.000901  0.000950     6  
1          0.001102  0.001269     6  
2          0.002581  0.002921     6  
3          0.003182  0.003790     6  
4          0.002615  0.003023     6  
5          0.003008  0.003563     6  
6          0.108523  0.088118     6  
7          0.106323  0.086212     6  
8          0.000681  0.000780     6  
9          0.000788  0.000944     6  
10         0.003938  0.002904     6  
11         0.003828  0.002842     6  

[12 rows x 26 columns]

,metric,method,n_dataset_clusters,stress_minus_clean_mean,CI95_low,CI95_high
0,bootstrap_cosine,AIME,3,-0.039785,-0.071039,-0.015233
1,bootstrap_cosine,HuberAIME,3,0.001345,-0.000943,0.004466
2,bootstrap_cosine,HuberRidgeAIME,3,0.001345,-0.000943,0.004467
3,bootstrap_cosine,LIME,3,0.011736,-0.001405,0.038016
4,bootstrap_cosine,RidgeAIME,3,-0.039786,-0.071040,-0.015233
5,bootstrap_cosine,SHAP_TreeSHAP,3,-0.002147,-0.007233,0.000836
6,noise_cosine,AIME,3,-0.004824,-0.013818,-0.000077
7,noise_cosine,HuberAIME,3,-0.001645,-0.004769,-0.000047
8,noise_cosine,HuberRidgeAIME,3,-0.001645,-0.004769,-0.000047
9,noise_cosine,LIME,3,0.004374,-0.004577,0.019527


[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/figures/Supplementary_Figure_S15_lime_treeshap_stress.png



## Experiment 5 — Moore–Penrose, truncated SVD, and ridge on the relevant matrices

The spectral comparison is now computed from the output design \(Y\) and the Huber-weighted design \(\sqrt W Y\). This makes the diagnostic directly relevant to the normal matrices in Eqs. (3), (5), and (7).

- Moore–Penrose: reciprocal filter for every nonzero singular direction.
- Truncated SVD: hard removal below a threshold.
- Ridge: continuous damping \(s/(s^2+\lambda)\) and strict positive definiteness of the regularized normal matrix.

Thus, adding \(\lambda I_C\) is not equivalent to merely invoking `pinv`, and it is not the same as hard singular-value truncation.


In [14]:

# %% [Experiment 5 and reporting clarifications]
def spectral_filters(s, lam=DEFAULT_RIDGE_LAMBDA, rcond=1e-3):
    s = np.asarray(s, float)
    smax = max(float(s.max()), 1e-12)
    return pd.DataFrame({
        "singular_value": s,
        "normalized_singular_value": s / smax,
        "pseudoinverse_filter": np.where(s > 1e-12 * smax, 1 / np.maximum(s, 1e-300), 0),
        "truncated_svd_filter": np.where(s > rcond * smax, 1 / np.maximum(s, 1e-300), 0),
        "ridge_filter": s / (s ** 2 + lam),
    })

def run_spectral_diagnostics():
    out = ADDITIONAL_DATADIR / "additional_validation_spectral_filter_diagnostics.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    for dataset in DATASETS:
        X, y, meta = standardized_dataset(dataset, ADDITIONAL_SEED)
        model, train_idx, exp_idx, Y_exp, info = fit_clean_blackbox(
            X, y, "lgbm", ADDITIONAL_SEED + 500
        )
        X_exp = X[exp_idx]
        y_exp = y[exp_idx]
        X_clean_aug, X_stress_aug, out_mask, _, _, stress_info = build_stress_pair(
            X_exp, 0.10, 0.50, ADDITIONAL_SEED + 501
        )
        fit_idx, _ = split_explanation_rows(y_exp, ADDITIONAL_SEED + 502)
        A, diag, state = fit_inverse_map(
            X_stress_aug[fit_idx], Y_exp[fit_idx],
            "HuberRidgeAIME", DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
            RESIDUAL_SCALE_MODE,
        )
        for design_name, design in [
            ("Y_unweighted", Y_exp[fit_idx]),
            ("sqrtW_Y_huber", Y_exp[fit_idx] * np.sqrt(state["weights"])[:, None]),
        ]:
            s = np.linalg.svd(design, compute_uv=False, full_matrices=False)
            f = spectral_filters(s)
            for rank_idx, r in f.reset_index(drop=True).iterrows():
                rows.append({
                    "dataset": dataset, "design": design_name,
                    "singular_index": rank_idx, **r.to_dict(),
                })
    df = pd.DataFrame(rows)
    save_csv(df, out)
    return df

def build_reporting_clarifications():
    df = pd.DataFrame([
        {
            "item": "Coefficient norm",
            "clarification": "log10||A||F is a regularization diagnostic, not a quality score; faithfulness is evaluated separately.",
        },
        {
            "item": "Identity notation",
            "clarification": "Use I_C whenever the dimension is explicit; I is only shorthand for the C x C identity.",
        },
        {
            "item": "IRLS iteration summaries",
            "clarification": "The real-data table reports the median across condition cells; the runtime table reports the mean across matrix-size and repeat runs.",
        },
        {
            "item": "Moore–Penrose versus ridge",
            "clarification": "pinv reciprocates nonzero singular values; ridge continuously damps all directions and makes Y^T W Y + lambda I_C strictly positive definite.",
        },
        {
            "item": "Truncated SVD versus ridge",
            "clarification": "truncated SVD applies a hard cutoff; ridge uses the continuous filter s/(s^2+lambda).",
        },
        {
            "item": "Huber threshold",
            "clarification": "delta is applied to row RMS input-space residuals, ||x_i-Ay_i||_2/sqrt(d), to permit cross-dataset comparison.",
        },
    ])
    save_table_bundle(
        df, "table_additional_validation_reporting_clarifications",
        "Clarifications for interpretation, notation, and apparently different iteration summaries.",
        "tab:additional_validation_reporting_clarifications",
    )
    return df

SPECTRAL_TSVD_RCOND = 1e-3
SPECTRAL_DESIGN_STYLES = {
    "Y_unweighted": {"color": "#1f77b4", "marker": "o", "label": "Unweighted Y"},
    "sqrtW_Y_huber": {"color": "#d62728", "marker": "s", "label": "Huber-weighted √W Y"},
}


def plot_spectral_manuscript_figure(df):
    dataset_labels = {
        "breast_cancer": "WDBC",
        "credit_approval": "Credit",
        "har": "HAR",
    }
    groups = [
        (dataset, design)
        for dataset in DATASETS
        for design in ["Y_unweighted", "sqrtW_Y_huber"]
        if len(df[(df["dataset"] == dataset) & (df["design"] == design)])
    ]
    positions = np.arange(len(groups), dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.0), dpi=220)
    seen_designs = set()
    for base_x, (dataset, design) in zip(positions, groups):
        group = df[(df["dataset"] == dataset) & (df["design"] == design)].copy()
        group = group.sort_values("singular_index")
        offsets = np.linspace(-0.13, 0.13, len(group)) if len(group) > 1 else np.asarray([0.0])
        style = SPECTRAL_DESIGN_STYLES[design]
        label = style["label"] if design not in seen_designs else None
        seen_designs.add(design)
        axes[0].scatter(
            base_x + offsets,
            group["normalized_singular_value"].to_numpy(float),
            color=style["color"],
            marker=style["marker"],
            s=35,
            alpha=0.85,
            label=label,
        )
        pinv = group["pseudoinverse_filter"].to_numpy(float)
        ridge = group["ridge_filter"].to_numpy(float)
        attenuation_bp = np.where(
            pinv > 0,
            10000.0 * (1.0 - ridge / pinv),
            np.nan,
        )
        axes[1].scatter(
            base_x + offsets,
            attenuation_bp,
            color=style["color"],
            marker=style["marker"],
            s=35,
            alpha=0.85,
        )

    tick_labels = [
        f"{dataset_labels.get(dataset, dataset)}\n"
        + ("Y" if design == "Y_unweighted" else "√W Y")
        for dataset, design in groups
    ]
    for ax in axes:
        ax.set_xticks(positions)
        ax.set_xticklabels(tick_labels)
        ax.grid(axis="y", alpha=0.20, linewidth=0.6)
    axes[0].set_yscale("log")
    axes[0].axhline(
        SPECTRAL_TSVD_RCOND,
        color="0.35",
        linestyle="--",
        linewidth=1,
        label="tSVD cutoff (rcond=10⁻³)",
    )
    axes[0].set_ylabel("Normalized singular value")
    axes[0].set_title("Output-design singular spectra")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].set_ylabel("Ridge attenuation relative to pinv (basis points)")
    axes[1].set_title("Continuous ridge damping at λ=0.01")
    for panel_label, ax in zip("ab", axes):
        ax.text(
            -0.10,
            1.05,
            panel_label,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="top",
        )
    if bool((df["normalized_singular_value"] > SPECTRAL_TSVD_RCOND).all()):
        axes[0].text(
            0.02,
            0.04,
            "All observed directions are retained by tSVD at rcond=10⁻³",
            transform=axes[0].transAxes,
            fontsize=8,
            color="0.35",
        )
    fig.tight_layout()
    path = ADDITIONAL_FIGDIR / "Supplementary_Figure_S16_spectral_diagnostics.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path


ADDITIONAL_SPECTRAL = run_spectral_diagnostics()
ADDITIONAL_SPECTRAL_MANUSCRIPT_FIGURE = plot_spectral_manuscript_figure(ADDITIONAL_SPECTRAL)
ADDITIONAL_REPORTING_CLARIFICATIONS = build_reporting_clarifications()
display(ADDITIONAL_REPORTING_CLARIFICATIONS)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/additional_validation_spectral_filter_diagnostics.csv
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/figures/Supplementary_Figure_S16_spectral_diagnostics.png
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/table_additional_validation_reporting_clarifications.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/tables/table_additional_validation_reporting_clarifications.tex


,item,clarification
0,Coefficient norm,"log10||A||F is a regularization diagnostic, no..."
1,Identity notation,Use I_C whenever the dimension is explicit; I ...
2,IRLS iteration summaries,The real-data table reports the median across ...
3,Moore–Penrose versus ridge,pinv reciprocates nonzero singular values; rid...
4,Truncated SVD versus ridge,truncated SVD applies a hard cutoff; ridge use...
5,Huber threshold,delta is applied to row RMS input-space residu...


## Validation gates, final package, provenance, and manuscript checklist

The final cell fails loudly if equation orientation, baseline completeness, required methods, or output inventory is incomplete. It then writes a machine-readable validation report, environment manifest, provenance table, manual manuscript checklist, and SHA-256 artifact manifest.


In [15]:
# %% [validation gates and final package]
def write_reporting_notes():
    text = """# Additional validation reporting notes

- Use the main reproducibility notebook as the source for the main real-data protocol.
- Use the fixed-black-box experiment to isolate the effect of contamination without refitting the predictor.
- Treat coefficient norm as a regularization diagnostic, not as an explanation-quality score.
- Report known-ground-truth faithfulness separately from stability diagnostics.
- Report lambda=0.01, delta=1.0, and the row-RMS residual definition with the sensitivity results.
- Keep results from distinct end-to-end protocols separate in the manuscript and supplementary material.
"""
    path = ADDITIONAL_OUTDIR / "additional_validation_reporting_notes.md"
    path.write_text(text, encoding="utf-8")
    return path

def write_readme():
    text = f"""# HuberRidgeAIME additional robustness and faithfulness outputs

Pipeline version: `{PIPELINE_VERSION}`

## Scientific scope
- All AIME-family results use `X ≈ Y A^T`.
- Fixed-black-box contamination and end-to-end stress protocols are reported separately.
- Huber weights use row-RMS input-space residuals.
- True irrelevant permutation decoys are separated from correlated clones.
- Coefficient norm is diagnostic only; ground-truth faithfulness is evaluated separately.
- Confidence intervals bootstrap condition clusters after averaging repeated seeds.
- Baseline execution uses standard LIME and native LightGBM TreeSHAP.
- Spectral diagnostics use `Y` and `sqrt(W)Y`.
- Manuscript-ready Supplementary Figures S11--S16 are generated directly from validated raw tables.
- Small horizontal offsets in the known-ground-truth figure are display-only; y-values and nominal outlier rates are unchanged.
- Nonzero x positions in the zero-operator control are display-scaled by plus or minus 4% to reveal overlapping methods; the underlying coefficient norms are unchanged.
- LIME/TreeSHAP comparisons use categorical method positions, and spectral diagnostics separate singular directions horizontally within categorical dataset/design groups.

## Archive note
This directory contains generated outputs. It is intentionally excluded from the source repository and can be regenerated by executing this notebook.
"""
    path = ADDITIONAL_OUTDIR / "README_additional_validation.md"
    path.write_text(text, encoding="utf-8")
    return path

def build_provenance():
    rows = [
        {
            "artifact": "table_additional_validation_hra_vs_ridge_cluster_ci.tex",
            "validation_scope": "HRA versus RidgeAIME with condition-cluster uncertainty",
            "source": (
                "additional_validation_hra_vs_ridge_fixed_black_box_raw.csv; "
                "additional_validation_hra_vs_ridge_end_to_end_raw.csv"
            ),
        },
        {
            "artifact": "table_additional_validation_hra_weight_diagnostics.tex",
            "validation_scope": "Huber row-weight diagnostics",
            "source": "additional-validation HRA-versus-Ridge raw files",
        },
        {
            "artifact": "table_additional_validation_synthetic_ground_truth_ci.tex",
            "validation_scope": "Known-ground-truth support and reconstruction faithfulness",
            "source": "additional_validation_synthetic_ground_truth_raw.csv",
        },
        {
            "artifact": "table_additional_validation_norm_is_not_quality.tex",
            "validation_scope": "Zero-operator control",
            "source": "additional_validation_synthetic_ground_truth_raw.csv",
        },
        {
            "artifact": "table_additional_validation_lambda_delta_sensitivity.tex",
            "validation_scope": "Ridge and Huber tuning sensitivity",
            "source": "additional_validation_lambda_delta_sensitivity_raw.csv",
        },
        {
            "artifact": "table_additional_validation_lime_shap_stress*.tex",
            "validation_scope": "LIME and TreeSHAP stress comparison",
            "source": "additional_validation_lime_shap_stress_raw.csv",
        },
        {
            "artifact": "table_additional_validation_dataset_diagnostics.tex",
            "validation_scope": "Dataset diagnostics",
            "source": "public dataset loaders",
        },
        {
            "artifact": "fig_additional_validation_spectral_*.png",
            "validation_scope": "Pseudoinverse, truncated-SVD, and ridge spectral diagnostics",
            "source": "additional_validation_spectral_filter_diagnostics.csv",
        },
        {
            "artifact": "Supplementary_Figure_S12_known_ground_truth.png",
            "validation_scope": "Manuscript-ready known-ground-truth comparison",
            "source": "additional_validation_synthetic_ground_truth_raw.csv",
        },
        {
            "artifact": "Supplementary_Figure_S11_fixed_black_box.png",
            "validation_scope": "Manuscript-ready fixed-black-box comparison",
            "source": "table_additional_validation_hra_vs_ridge_cluster_ci.csv",
        },
        {
            "artifact": "Supplementary_Figure_S13_zero_operator_control.png",
            "validation_scope": "Manuscript-ready zero-operator control",
            "source": "additional_validation_synthetic_ground_truth_raw.csv",
        },
        {
            "artifact": "Supplementary_Figure_S14_lambda_delta_sensitivity.png",
            "validation_scope": "Manuscript-ready tuning sensitivity",
            "source": "additional_validation_lambda_delta_sensitivity_raw.csv",
        },
        {
            "artifact": "Supplementary_Figure_S15_lime_treeshap_stress.png",
            "validation_scope": "Manuscript-ready LIME and TreeSHAP stress comparison",
            "source": "table_additional_validation_lime_shap_stress_changes.csv",
        },
        {
            "artifact": "Supplementary_Figure_S16_spectral_diagnostics.png",
            "validation_scope": "Manuscript-ready output-design spectral diagnostics",
            "source": "additional_validation_spectral_filter_diagnostics.csv",
        },
    ]
    df = pd.DataFrame(rows)
    save_csv(
        df,
        ADDITIONAL_DATADIR / "additional_validation_figure_table_provenance.csv",
    )
    return df


def write_run_configuration():
    config = {
        "pipeline_version": PIPELINE_VERSION,
        "seed": ADDITIONAL_SEED,
        "quick_test": QUICK_TEST,
        "force_recompute": FORCE_RECOMPUTE,
        "run_end_to_end": RUN_END_TO_END,
        "run_lime_treeshap": RUN_BASELINES,
        "datasets": DATASETS,
        "learners": LEARNERS,
        "max_n_per_dataset": MAX_N_PER_DATASET,
        "max_clones": MAX_CLONES,
        "default_ridge_lambda": DEFAULT_RIDGE_LAMBDA,
        "default_huber_delta": DEFAULT_HUBER_DELTA,
        "residual_scale_mode": RESIDUAL_SCALE_MODE,
        "top_k": TOP_K,
        "real_data_repeats_fixed": ADDITIONAL_REPEATS,
        "real_data_repeats_end_to_end": E2E_REPEATS,
        "outlier_levels": ADDITIONAL_OUTLIER_LEVELS,
        "clone_fractions": ADDITIONAL_CLONE_FRACS,
        "bootstrap_refits": ADDITIONAL_BOOTSTRAPS,
        "cluster_bootstrap_resamples": ADDITIONAL_CI_RESAMPLES,
        "irrelevant_decoy_trials": IRRELEVANT_DECOY_TRIALS,
        "irrelevant_decoy_count": IRRELEVANT_DECOY_COUNT,
        "synthetic_repeats": SYN_REPEATS,
        "synthetic_n": SYN_N,
        "synthetic_d": SYN_D,
        "synthetic_k": SYN_K,
        "synthetic_rhos": SYN_RHOS,
        "synthetic_outlier_levels": SYN_OUTLIER_LEVELS,
        "lambda_grid": LAMBDA_GRID,
        "delta_grid": DELTA_GRID,
        "baseline_datasets": BASELINE_DATASETS,
        "baseline_learner": BASELINE_LEARNER,
        "baseline_repeats": BASELINE_REPEATS,
        "baseline_conditions": BASELINE_CONDITIONS,
        "baseline_eval_n": BASELINE_EVAL_N,
        "lime_num_samples": LIME_NUM_SAMPLES,
        "lime_num_features": LIME_NUM_FEATURES,
        "shap_implementation": "LightGBM native TreeSHAP pred_contrib=True",
        "manuscript_ready_figures": [
            "Supplementary_Figure_S12_known_ground_truth.png",
            "Supplementary_Figure_S11_fixed_black_box.png",
            "Supplementary_Figure_S13_zero_operator_control.png",
            "Supplementary_Figure_S14_lambda_delta_sensitivity.png",
            "Supplementary_Figure_S15_lime_treeshap_stress.png",
            "Supplementary_Figure_S16_spectral_diagnostics.png",
        ],
        "synthetic_plot_horizontal_offsets": SYNTHETIC_PLOT_HORIZONTAL_OFFSETS,
        "zero_control_x_display_factors": ZERO_CONTROL_X_DISPLAY_FACTORS,
        "spectral_tsvd_rcond": SPECTRAL_TSVD_RCOND,
    }
    path = ADDITIONAL_OUTDIR / "additional_validation_run_configuration.json"
    path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
    return path

def write_environment_manifest():
    rows=[]
    for mod in ["numpy","pandas","scipy","sklearn","matplotlib","lightgbm"]:
        try:
            m=__import__(mod); version=getattr(m,"__version__","available")
        except Exception as e: version=f"not available: {e}"
        rows.append({"package":mod,"version":version})
    rows += [
        {"package":"python","version":sys.version.replace("\n"," ")},
        {"package":"platform","version":platform.platform()},
        {"package":"LIME implementation","version":LIME_IMPLEMENTATION},
        {"package":"SHAP implementation","version":"LightGBM native TreeSHAP pred_contrib=True"},
        {"package":"pipeline_version","version":PIPELINE_VERSION},
    ]
    df=pd.DataFrame(rows); save_csv(df,ADDITIONAL_DATADIR/"additional_validation_environment_manifest.csv"); return df

def _finite_columns(df, columns, mask=None):
    if mask is not None:
        df = df.loc[mask]
    if df.empty:
        return False
    return bool(np.isfinite(df[columns].to_numpy(dtype=float)).all())

def validate_outputs():
    checks = {}

    expected_fixed = (
        len(DATASETS) * len(LEARNERS) * ADDITIONAL_REPEATS *
        len(ADDITIONAL_OUTLIER_LEVELS) * len(ADDITIONAL_CLONE_FRACS) * 2
    )
    expected_e2e = (
        len(DATASETS) * len(LEARNERS) * E2E_REPEATS *
        len(ADDITIONAL_OUTLIER_LEVELS) * len(ADDITIONAL_CLONE_FRACS) * 2
        if RUN_END_TO_END else 0
    )
    expected_synthetic = (
        SYN_REPEATS * len(SYN_RHOS) * len(SYN_OUTLIER_LEVELS) * 5
    )
    expected_sensitivity = (
        len(DATASETS) * len(LEARNERS) * SENSITIVITY_REPEATS *
        len(LAMBDA_GRID) * (1 + len(DELTA_GRID))
    )
    expected_baselines = (
        len(BASELINE_DATASETS) * BASELINE_REPEATS *
        len(BASELINE_CONDITIONS) * 6 if RUN_BASELINES else 0
    )

    checks["inverse_orientation_all_real_rows"] = bool(
        (ADDITIONAL_REAL_RAW["orientation"] == "X ~= Y A^T").all()
    )
    checks["real_data_no_errors"] = bool(
        (ADDITIONAL_REAL_RAW["error"].fillna("") == "").all()
    )
    checks["fixed_protocol_expected_rows"] = len(ADDITIONAL_FIXED_RAW) == expected_fixed
    checks["end_to_end_expected_rows"] = (
        (not RUN_END_TO_END) or len(ADDITIONAL_E2E_RAW) == expected_e2e
    )
    checks["real_data_has_both_methods"] = {
        "RidgeAIME", "HuberRidgeAIME"
    }.issubset(set(ADDITIONAL_REAL_RAW["method"]))
    checks["real_data_has_fixed_protocol"] = (
        "fixed_black_box_contamination" in set(ADDITIONAL_REAL_RAW["experiment_mode"])
    )
    checks["real_data_has_end_to_end_protocol"] = (
        (not RUN_END_TO_END) or
        ("end_to_end_original_stress" in set(ADDITIONAL_REAL_RAW["experiment_mode"]))
    )
    checks["real_data_key_metrics_finite"] = _finite_columns(
        ADDITIONAL_REAL_RAW,
        [
            "clean_target_reconstruction_cosine",
            "operator_cosine_flat",
            "irrelevant_decoy_mass_ratio",
            "irrelevant_decoy_topk_infiltration",
            "log10_cond_reg",
            "log10_coef_norm",
        ],
    )
    checks["cluster_ci_nonempty_and_finite"] = (
        len(ADDITIONAL_REAL_CI) > 0 and
        _finite_columns(
            ADDITIONAL_REAL_CI,
            ["HRA_minus_Ridge_cluster_mean", "CI95_low", "CI95_high"]
        )
    )

    checks["synthetic_expected_rows"] = len(ADDITIONAL_SYNTH_RAW) == expected_synthetic
    checks["synthetic_has_zero_control"] = "ZeroOperator" in set(ADDITIONAL_SYNTH_RAW["method"])
    checks["synthetic_has_hra_and_ridge"] = {
        "RidgeAIME", "HuberRidgeAIME"
    }.issubset(set(ADDITIONAL_SYNTH_RAW["method"]))
    checks["synthetic_key_metrics_finite_nonzero_methods"] = _finite_columns(
        ADDITIONAL_SYNTH_RAW,
        [
            "support_average_precision", "f1_at_k", "truth_cosine",
            "sign_agreement_relevant", "clean_target_reconstruction_r2",
            "clean_target_reconstruction_cosine", "log10_coef_norm",
        ],
        ADDITIONAL_SYNTH_RAW["method"] != "ZeroOperator",
    )

    checks["sensitivity_expected_rows"] = len(ADDITIONAL_SENS_RAW) == expected_sensitivity
    checks["sensitivity_has_true_decoy_metrics"] = {
        "irrelevant_decoy_mass_ratio", "irrelevant_decoy_topk_infiltration"
    }.issubset(ADDITIONAL_SENS_RAW.columns)
    checks["sensitivity_key_metrics_finite"] = _finite_columns(
        ADDITIONAL_SENS_RAW,
        [
            "log10_cond_reg", "log10_coef_norm",
            "clean_target_reconstruction_cosine", "operator_cosine_flat",
            "irrelevant_decoy_mass_ratio", "clone_mass_ratio",
        ],
    )

    if RUN_BASELINES:
        checks["full_run_has_genuine_lightgbm"] = QUICK_TEST or LGBM_AVAILABLE
        checks["full_run_has_standard_lime"] = QUICK_TEST or LIME_PACKAGE_AVAILABLE
        checks["baseline_expected_rows"] = len(ADDITIONAL_BASELINE_RAW) == expected_baselines
        checks["baseline_no_errors"] = bool(
            (ADDITIONAL_BASELINE_RAW["error"].fillna("") == "").all()
        )
        checks["baseline_has_all_methods"] = {
            "AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME",
            "LIME", "SHAP_TreeSHAP"
        }.issubset(set(ADDITIONAL_BASELINE_RAW["method"]))
        checks["baseline_key_metrics_finite"] = _finite_columns(
            ADDITIONAL_BASELINE_RAW,
            [
                "bootstrap_cosine", "bootstrap_spearman", "bootstrap_topk",
                "noise_cosine", "noise_spearman", "noise_topk",
                "added_clone_mass", "explain_time_sec",
            ],
        )
    else:
        checks["full_run_has_genuine_lightgbm"] = True
        checks["full_run_has_standard_lime"] = True
        checks["baseline_expected_rows"] = True
        checks["baseline_no_errors"] = True
        checks["baseline_has_all_methods"] = True
        checks["baseline_key_metrics_finite"] = True

    checks["spectral_has_relevant_designs"] = set(ADDITIONAL_SPECTRAL["design"]).issuperset(
        {"Y_unweighted", "sqrtW_Y_huber"}
    )
    checks["spectral_values_finite"] = _finite_columns(
        ADDITIONAL_SPECTRAL,
        [
            "singular_value", "normalized_singular_value",
            "pseudoinverse_filter", "truncated_svd_filter", "ridge_filter",
        ],
    )

    manuscript_figure_names = [
        "Supplementary_Figure_S11_fixed_black_box.png",
        "Supplementary_Figure_S12_known_ground_truth.png",
        "Supplementary_Figure_S13_zero_operator_control.png",
        "Supplementary_Figure_S14_lambda_delta_sensitivity.png",
        "Supplementary_Figure_S16_spectral_diagnostics.png",
    ]
    if RUN_BASELINES:
        manuscript_figure_names.append(
            "Supplementary_Figure_S15_lime_treeshap_stress.png"
        )
    checks["manuscript_ready_figures_generated"] = all(
        (ADDITIONAL_FIGDIR / name).is_file() for name in manuscript_figure_names
    )

    failed = [k for k, v in checks.items() if not bool(v)]
    report = {
        "pipeline_version": PIPELINE_VERSION,
        "quick_test": QUICK_TEST,
        "expected_rows": {
            "fixed": expected_fixed,
            "end_to_end": expected_e2e,
            "synthetic": expected_synthetic,
            "sensitivity": expected_sensitivity,
            "baselines": expected_baselines,
        },
        "observed_rows": {
            "fixed": len(ADDITIONAL_FIXED_RAW),
            "end_to_end": len(ADDITIONAL_E2E_RAW),
            "synthetic": len(ADDITIONAL_SYNTH_RAW),
            "sensitivity": len(ADDITIONAL_SENS_RAW),
            "baselines": len(ADDITIONAL_BASELINE_RAW),
        },
        "checks": checks,
        "failed": failed,
    }
    path = ADDITIONAL_OUTDIR / "additional_validation_validation_report.json"
    path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    if failed:
        raise AssertionError(f"Validation failed: {failed}")
    print("All validation gates passed.")
    return report

def write_requirements():
    text="""numpy\npandas\nscipy\nscikit-learn\nmatplotlib\nlightgbm\nlime\nnbformat\n"""
    path=ADDITIONAL_OUTDIR/"requirements_additional_validation.txt"; path.write_text(text,encoding="utf-8"); return path

def build_manifest_and_zip():
    artifacts=[]
    for p in sorted(ADDITIONAL_OUTDIR.rglob("*")):
        if p.is_file() and not p.name.endswith(".zip") and p.name != "additional_validation_artifact_manifest.csv":
            artifacts.append({"relative_path":str(p.relative_to(ADDITIONAL_OUTDIR)),"bytes":p.stat().st_size,"sha256":file_sha256(p)})
    manifest=pd.DataFrame(artifacts); save_csv(manifest,ADDITIONAL_OUTDIR/"additional_validation_artifact_manifest.csv")
    zip_path=ADDITIONAL_OUTDIR/"HuberRidgeAIME_Additional_Robustness_Faithfulness_outputs.zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
        for p in ADDITIONAL_OUTDIR.rglob("*"):
            if p.is_file() and p.resolve()!=zip_path.resolve(): zf.write(p,p.relative_to(ADDITIONAL_OUTDIR))
    print("[zip]",zip_path); return zip_path

CHECKLIST_PATH=write_reporting_notes(); README_PATH=write_readme(); REQUIREMENTS_PATH=write_requirements(); RUN_CONFIG_PATH=write_run_configuration()
ADDITIONAL_PROVENANCE=build_provenance(); ADDITIONAL_ENVIRONMENT=write_environment_manifest(); ADDITIONAL_VALIDATION=validate_outputs()
ZIP_PATH=build_manifest_and_zip()

print("\nOutput inventory")
for folder in [ADDITIONAL_DATADIR,ADDITIONAL_TABLEDIR,ADDITIONAL_FIGDIR,ADDITIONAL_LOGDIR]:
    files=sorted(p.name for p in folder.glob("*") if p.is_file())
    print(f"{folder.name}/: {len(files)} files")
    for name in files: print(" -",name)
print("ZIP:",ZIP_PATH)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/additional_validation_figure_table_provenance.csv
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/data/additional_validation_environment_manifest.csv
All validation gates passed.
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/additional_validation_artifact_manifest.csv
[zip] /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/additional_validation_manuscript_figures/HuberRidgeAIME_Additional_Robustness_Faithfulness_outputs.zip

Output inventory
data/: 18 files
 - additional_validation_environment_manifest.csv
 - additional_validation_figure_table_provenance.csv
 - additional_validation_hra_vs_ridge_end_to_end_raw.csv
 - additional_validation_hra_vs_ridge_fixed_black_box_raw.csv
 - additional_vali